# Amazon review-log analysis — ratings & verified purchases

This notebook analyses the raw Amazon interaction logs used by FlowRec and looks at:

1. **Rating distribution** — how the 1–5 star ratings are spread.
2. **`verified_purchase` distribution** — the share of verified vs. non-verified reviews.
3. **Joint distribution** — how the verified flag relates to the star rating.

**Data.** We read the "dataframe format" parquet logs in `data/raw/dataframe_format/`.
Three review logs are available; they use slightly different column names, which the
loader normalises to a common schema (`rating`, `verified`):

| dataset | rows | rating col | verified col | notes |
|---|---|---|---|---|
| `instruments_2018` | ~1.5 M | `overall` | `verified` | verified varies (default) |
| `beauty_2023` | ~24 M | `rating` | `verified_purchase` | verified varies, large |
| `beauty_2014` | ~2.0 M | `overall` | `verified` | **all** reviews verified — joint is degenerate |

Switch datasets by changing the `DATASET` variable in the loader cell.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

# --- Palette (validated data-viz reference, light surface) ---------------------
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"

# Verified status = categorical identity (blue / orange are far apart for CVD).
C_VERIFIED, C_UNVERIFIED, C_UNKNOWN = "#2a78d6", "#eb6834", "#898781"
COLOR_BY_LABEL = {"Verified": C_VERIFIED, "Not verified": C_UNVERIFIED, "Unknown": C_UNKNOWN}

# Rating = ordinal magnitude -> single-hue blue ramp, light (1★) -> dark (5★).
RATING_RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]
RATINGS = np.array([1, 2, 3, 4, 5])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.linewidth": 1.0,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 1.0,
    "axes.spines.top": False, "axes.spines.right": False,
    "text.color": INK, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 11, "figure.dpi": 110,
})

# Compact count formatter for axes (1.2M / 340k / 42).
kfmt = FuncFormatter(lambda v, _: f"{v/1e6:.1f}M" if v >= 1e6 else (f"{v/1e3:.0f}k" if v >= 1e3 else f"{v:.0f}"))

def ink_on(hex_bg):
    """Pick white or dark ink for a label placed on a coloured fill."""
    r, g, b = (int(hex_bg[i:i + 2], 16) for i in (1, 3, 5))
    return "#ffffff" if (0.299 * r + 0.587 * g + 0.114 * b) / 255 < 0.6 else INK


In [ ]:
# --- Dataset registry: name -> (parquet file, rating column, verified column) ---
DATASETS = {
    "instruments_2018": ("reviews_instruments_2018.parquet", "overall", "verified"),
    "beauty_2023":      ("reviews_beauty_2023.parquet",      "rating",  "verified_purchase"),
    "beauty_2014":      ("reviews_beauty_2014.parquet",      "overall", "verified"),
}

def _resolve_data_dir():
    for p in (Path("../data/raw/dataframe_format"),
              Path("data/raw/dataframe_format"),
              Path("../../data/raw/dataframe_format")):
        if p.exists():
            return p
    raise FileNotFoundError("Could not locate data/raw/dataframe_format")

DATA_DIR = _resolve_data_dir()

def load_reviews(name):
    """Load a review log and normalise to columns: `rating` (Int64 1-5) and
    `verified` (nullable boolean; nulls kept as <NA>). Only the two needed
    columns are read, so even the 24 M-row log loads quickly."""
    fname, rating_col, verified_col = DATASETS[name]
    df = (pd.read_parquet(DATA_DIR / fname, columns=[rating_col, verified_col])
            .rename(columns={rating_col: "rating", verified_col: "verified"}))
    df["rating"] = df["rating"].round().astype("Int64")
    df["verified"] = df["verified"].astype("boolean")
    return df

# >>> Change this to analyse a different log. <<<
DATASET = "beauty_2023"
reviews = load_reviews(DATASET)

# Human-readable verified label (keeps nulls as an explicit "Unknown" bucket).
reviews["verified_label"] = (
    reviews["verified"].map({True: "Verified", False: "Not verified"})
    .where(reviews["verified"].notna(), "Unknown")
)

print(f"Loaded '{DATASET}':  {len(reviews):,} reviews")
reviews.head()


In [ ]:
# --- Quick numeric overview ----------------------------------------------------
n = len(reviews)
print(f"Rows: {n:,}")
print(f"Mean rating: {reviews['rating'].mean():.3f}   median: {reviews['rating'].median():.0f}\n")

print("Rating counts:")
for r, c in reviews["rating"].value_counts().reindex(RATINGS, fill_value=0).items():
    print(f"  {r}★  {c:>12,}  ({c / n:6.2%})")

print("\nverified_purchase:")
for lab, c in reviews["verified_label"].value_counts().items():
    print(f"  {lab:<12} {c:>12,}  ({c / n:6.2%})")


## 1. Rating distribution

How the 1–5 star ratings are spread across the log. Amazon ratings are famously
**J-shaped** — heavily skewed toward 5★, with a smaller bump at 1★.

In [ ]:
# --- Marginal: rating distribution ---------------------------------------------
rating_counts = reviews["rating"].value_counts().reindex(RATINGS, fill_value=0).to_numpy()
rating_share = rating_counts / rating_counts.sum()

fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.bar(RATINGS, rating_counts, width=0.72, color=RATING_RAMP,
       edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.set_title(f"Rating distribution — {DATASET}", fontweight="bold", pad=12, loc="left")
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
ax.set_xticks(RATINGS)
ax.margins(y=0.15)
ax.yaxis.set_major_formatter(kfmt)
ax.grid(axis="x", visible=False)
for x, c, s in zip(RATINGS, rating_counts, rating_share):
    ax.text(x, c, f"{c:,}\n{s:.1%}", ha="center", va="bottom", fontsize=9, color=INK_2)
fig.tight_layout()
plt.show()


## 2. `verified_purchase` distribution

Share of reviews flagged as verified purchases. Any `null` flags are kept as an
explicit **Unknown** bucket rather than silently dropped. (Note: in the
`beauty_2014` log every review is verified, so this reduces to a single bar.)

In [ ]:
# --- Marginal: verified_purchase distribution ----------------------------------
present = [lab for lab in ("Verified", "Not verified", "Unknown")
           if (reviews["verified_label"] == lab).any()]
counts = np.array([(reviews["verified_label"] == lab).sum() for lab in present])

fig, ax = plt.subplots(figsize=(7.5, 2.4 + 0.35 * len(present)))
ypos = np.arange(len(present))[::-1]  # first label on top
ax.barh(ypos, counts, height=0.6, color=[COLOR_BY_LABEL[l] for l in present],
        edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.set_yticks(ypos)
ax.set_yticklabels(present)
ax.set_title(f"Verified-purchase distribution — {DATASET}", fontweight="bold", pad=12, loc="left")
ax.set_xlabel("Number of reviews")
ax.xaxis.set_major_formatter(kfmt)
ax.grid(axis="y", visible=False)
ax.margins(x=0.18)
for y, c in zip(ypos, counts):
    ax.text(c, y, f"  {c:,}  ({c / n:.1%})", va="center", ha="left", fontsize=10, color=INK_2)
fig.tight_layout()
plt.show()


## 3. Joint distribution — rating × verified

Two views of the same crosstab:

- **Left — joint counts:** absolute number of reviews for each (rating, verified) pair.
- **Right — verified share within each rating:** each bar normalised to 100%, so you
  can read *P(verified | rating)* directly and see whether higher-rated reviews are
  more likely to be verified.

In [ ]:
# --- Joint: rating x verified --------------------------------------------------
ct = pd.crosstab(reviews["rating"], reviews["verified_label"]).reindex(RATINGS, fill_value=0)
cats = [c for c in ("Verified", "Not verified", "Unknown") if c in ct.columns]
ct = ct[cats]
prop = ct.div(ct.sum(axis=1), axis=0)  # row-normalised: share within each rating

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.7))

# Left: grouped counts.
w = 0.8 / len(cats)
for i, cat in enumerate(cats):
    ax1.bar(RATINGS + (i - (len(cats) - 1) / 2) * w, ct[cat].to_numpy(), width=w * 0.9,
            color=COLOR_BY_LABEL[cat], edgecolor=SURFACE, linewidth=1.0, label=cat, zorder=3)
ax1.set_title("Joint counts — rating × verified", fontweight="bold", pad=10, loc="left")
ax1.set_xlabel("Star rating")
ax1.set_ylabel("Number of reviews")
ax1.set_xticks(RATINGS)
ax1.yaxis.set_major_formatter(kfmt)
ax1.grid(axis="x", visible=False)
ax1.legend(frameon=False, loc="upper left")

# Right: 100%-stacked verified share within each rating.
bottom = np.zeros(len(RATINGS))
for cat in cats:
    seg = prop[cat].to_numpy()
    ax2.bar(RATINGS, seg, bottom=bottom, width=0.72, color=COLOR_BY_LABEL[cat],
            edgecolor=SURFACE, linewidth=1.2, zorder=3)
    for x, val, b in zip(RATINGS, seg, bottom):
        if val > 0.05:
            ax2.text(x, b + val / 2, f"{val:.0%}", ha="center", va="center",
                     color=ink_on(COLOR_BY_LABEL[cat]), fontsize=8.5, fontweight="bold")
    bottom += seg
ax2.set_title("Verified share within each rating", fontweight="bold", pad=10, loc="left")
ax2.set_xlabel("Star rating")
ax2.set_ylabel("Share of reviews")
ax2.set_xticks(RATINGS)
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
ax2.grid(visible=False)

fig.tight_layout()
plt.show()


In [ ]:
# --- Joint crosstab + summary statistics ---------------------------------------
print("Joint counts (rating × verified):")
print(ct.to_string())

print("\nVerified rate by rating  P(verified | rating):")
if "Verified" in ct.columns:
    vrate = (ct["Verified"] / ct.sum(axis=1))
    print(vrate.round(3).to_string())

print("\nMean rating by verified status  E[rating | verified]:")
print(reviews.groupby("verified_label", observed=True)["rating"]
      .agg(["mean", "count"]).round(3).to_string())


## 4. Missing / unset values (data quality)

Are there interactions where the **rating** or the **`verified_purchase`** flag is
not set at all — empty, `NaN`, `null`, or out of the valid 1–5 range?

We check on the loaded log and then sweep every dataset:

- `rating` missing (`NaN`) or outside the 1–5 range,
- `verified` missing (`null`),
- rows missing **either** field, and rows missing **both**.

> **Where the missingness actually lives.** The `dataframe_format/*.parquet` logs
> analysed above are *already cleaned*, so we expect ~0 missing here. The real gaps
> are **upstream** — the last cell inspects the converted implicit-feedback `.jsonl`
> logs that FlowRec's `AmazonLogRewardModel` actually consumes, where the flag is
> genuinely absent. That absence is exactly why `AmazonLogRewardConfig` treats the
> rating/verified fields as *optional* with an `unobserved_reward` fallback.

In [ ]:
# --- Missing-value report for the loaded log -----------------------------------
def missingness_report(df, name):
    n = len(df)
    rating_na = df["rating"].isna()
    verified_na = df["verified"].isna()
    out_of_range = df["rating"].notna() & ((df["rating"] < 1) | (df["rating"] > 5))
    rows = [
        ("rating missing (NaN)",             int(rating_na.sum())),
        ("rating out of 1-5 range",          int(out_of_range.sum())),
        ("verified missing (null)",          int(verified_na.sum())),
        ("row missing rating OR verified",   int((rating_na | verified_na).sum())),
        ("row missing rating AND verified",  int((rating_na & verified_na).sum())),
    ]
    rep = pd.DataFrame(rows, columns=["check", "count"])
    rep["pct"] = (rep["count"] / n * 100).round(4)
    print(f"Missing-value report — {name}  (n={n:,})")
    print(rep.to_string(index=False))
    return rep

report = missingness_report(reviews, DATASET)

# --- Field-completeness bar (present vs missing) -------------------------------
C_PRESENT, C_MISSING = C_VERIFIED, "#d03b3b"   # blue = present, red = missing
FIELDS = ["rating", "verified"]
missing_frac = np.array([reviews[f].isna().mean() for f in FIELDS])
present_frac = 1 - missing_frac

fig, ax = plt.subplots(figsize=(8, 2.5))
ypos = np.arange(len(FIELDS))[::-1]
ax.barh(ypos, present_frac, color=C_PRESENT, edgecolor=SURFACE, linewidth=1.5,
        zorder=3, label="present")
ax.barh(ypos, missing_frac, left=present_frac, color=C_MISSING, edgecolor=SURFACE,
        linewidth=1.5, hatch="///", zorder=3, label="missing")
ax.set_yticks(ypos)
ax.set_yticklabels(FIELDS)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis="y", visible=False)
ax.set_title(f"Field completeness — {DATASET}", fontweight="bold", loc="left", pad=10)
for y, f, m in zip(ypos, FIELDS, missing_frac):
    ax.text(0.01, y, f"{1 - m:.2%} complete · {int(reviews[f].isna().sum()):,} missing",
            va="center", ha="left", color="white", fontsize=9, fontweight="bold")
ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(1, 1.02), ncol=2)
fig.tight_layout()
plt.show()


In [ ]:
# --- Sweep every parquet log for missing rating / verified ---------------------
sweep = []
for name in DATASETS:
    df = load_reviews(name)
    n = len(df)
    sweep.append({
        "dataset": name,
        "rows": n,
        "rating_missing": int(df["rating"].isna().sum()),
        "verified_missing": int(df["verified"].isna().sum()),
        "verified_true_%": round((df["verified"] == True).mean() * 100, 2),
        "verified_false_%": round((df["verified"] == False).mean() * 100, 2),
    })
sweep = pd.DataFrame(sweep)
print("Missingness across all parquet logs:")
print(sweep.to_string(index=False))


In [ ]:
# --- Upstream check: converted logs consumed by AmazonLogRewardModel -----------
# These *_converted_for_amazonreview.jsonl.gz logs are the implicit-feedback
# interactions the reward model reads. Unlike the cleaned parquet, the rating /
# verified_purchase fields here can be entirely unset (null) or synthetic.
import gzip
import json

RAW_DIR = DATA_DIR.parent  # data/raw
converted = sorted(RAW_DIR.glob("reviews_*converted_for_amazonreview.jsonl.gz"))

rows = []
for path in converted:
    n = rating_null = verified_null = 0
    rating_vals, verified_vals = set(), set()
    with gzip.open(path, "rt") as fh:
        for line in fh:
            d = json.loads(line)
            n += 1
            r = d.get("rating", d.get("overall"))
            v = d.get("verified_purchase", d.get("verified"))
            rating_null += r is None
            verified_null += v is None
            rating_vals.add(r)
            verified_vals.add(v)
    rows.append({
        "file": path.name.replace("_for_amazonreview.jsonl.gz", ""),
        "rows": n,
        "rating_null_%": round(rating_null / n * 100, 1),
        "verified_null_%": round(verified_null / n * 100, 1),
        "distinct_rating": sorted(x for x in rating_vals if x is not None),
        "distinct_verified": sorted(str(x) for x in verified_vals),
    })

upstream = pd.DataFrame(rows)
print("Rating / verified_purchase in the converted logs the reward model consumes:")
with pd.option_context("display.max_colwidth", 40):
    print(upstream.to_string(index=False))


### What the missingness check shows

- **The cleaned parquet logs have no unset values.** Across `instruments_2018`,
  `beauty_2023`, and `beauty_2014`, every row has a `rating` in 1–5 and a boolean
  `verified` flag — `0` missing on both. So for the modelling data there are **no
  interactions with an empty/NaN rating or verified flag**.
- **`verified` is structurally absent in the 2014 Amazon format.** The original
  `2014/reviews_Beauty.json` has no `verified` key at all — the field was only added
  in the 2018 dump. The parquet build backfills it as all-`True`, which is why
  `beauty_2014` shows `verified_false_% = 0`. Treat its "verified" column as *unknown*,
  not as a genuine 100%-verified signal.
- **The converted logs the reward model reads are where the flag is truly unset.**
  In the `*_uploaded_converted` implicit-feedback logs, `verified_purchase` is
  **100% null** and the rating is a synthetic constant — these are click/interaction
  logs, not rated reviews. `AmazonLogRewardModel` handles this gracefully: missing
  ratings/verified fall back to `unobserved_reward`, and the verified bonus is only
  applied when the flag is actually present.

## 5. Per-user reward distributions across reward modes

For a few random users (with their interaction prefix) we compute the reward of
**every item in the catalog** under each reward-generation mode and compare the
distributions:

| mode | description |
|---|---|
| `amazon_log / logged` | original scheme: normalized rating × verified bonus → `log1p`; unobserved items get `unobserved_reward` |
| `amazon_log / binary` | positive target → `positive_reward` (1.0), everything else → `unobserved_reward` |
| `amazon_log / item_centered` | multiplier `clip(exp(β·w·z))` around 1.0, where `z` is the item-mean-centered rating residual and `w` down-weights unverified evidence |
| `sasrec / <transform>` | tuned-SASRec next-item scores mapped to a positive reward by each `SASRecRewardConfig.transform`: `softplus`, `sigmoid`, `identity`, `exp`, `minmax_batch`, `rank_percentile` |
| `sasrec × ic / <transform>` | `MultiplicativeRewardModel`: the collaborative reward **multiplied** by the `item_centered` modulator, per transform |
| `sasrec + ic / <transform>` | `MixedRewardModel` (`scripts/run_tiger.py`, `reward.name="mix"`): batch_max-normalized weighted **sum** of the collaborative reward and the `item_centered` modulator, per transform |

**Setup.** This section always uses **instruments** (independent of the `DATASET`
variable above), because that is what the tuned SASRec checkpoint
`checkpoints/exp2-tuned-params/sasrec/instruments` was trained on. Data
preprocessing replicates `scripts/tiger_for_small_hard.py` (leave-one-out, 5-core,
`max_history_len=20`), plus the semantic-ID catalog filter from
`scripts/rewards_ablation.py` — filtering to the semantic-ID item set *before*
k-core/factorization reproduces the exact item IDs the checkpoint was trained on.
Users are drawn from the test split, so the prefix is the user's full history
minus the held-out target.

In [ ]:
# --- FlowRec setup: datamodule (tiger_for_small_hard preprocessing) + SASRec ---
import sys

import torch

PROJECT_ROOT = next(
    p for p in (Path.cwd(), Path.cwd().parent)
    if (p / "flowrec").exists() and (p / "data" / "raw").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.models.sasrec import SASRec, SASRecConfig

dm = AmazonReviewDataModule(
    AmazonReviewDataConfig(
        reviews_path=PROJECT_ROOT / "data/raw/dataframe_format/reviews_instruments_2018.parquet",
        metadata_path=PROJECT_ROOT / "data/raw/dataframe_format/meta_instruments_2018.parquet",
        # Filtering to the semantic-ID catalog before k-core / factorization
        # reproduces the item IDs the tuned SASRec checkpoint was trained on
        # (same recipe as scripts/rewards_ablation.py).
        semantic_id_path=PROJECT_ROOT / "datasets/semantic_ids/exp1-standard params/instruments/semantic_ids.json",
        drop_items_missing_semantic_id=True,
        item_key="asin",
        # Must stay None: this notebook's reward models read ratings from
        # batches, and the SASRec checkpoint below pins the exact item
        # catalog. Migrates to store-based lookups in the step-2 refactor.
        positive_criterion=None,
        split_strategy="leave_one_out",
        train_all_prefixes=True,
        min_user_interactions=5,
        min_item_interactions=5,
        max_history_len=20,
        deduplicate_consecutive=False,
        filter_cold_items_after_split=False,
        batch_size=128,
        eval_batch_size=64,
        num_workers=0,
        pin_memory=False,
        persistent_workers=False,
        offset_semantic_ids_by_one=True,
        seed=43,
    )
)
dm.setup()
print(f"datamodule: users={dm.num_users:,}  items={dm.num_items:,}  "
      f"vocab={dm.item_vocab_size}  test examples={len(dm.test_dataset):,}")

SASREC_CKPT = PROJECT_ROOT / "checkpoints/exp2-tuned-params/sasrec/instruments/tuned_sasrec_instruments.pt"
payload = torch.load(SASREC_CKPT, map_location="cpu", weights_only=False)
assert payload["model_config"]["item_vocab_size"] == dm.item_vocab_size, (
    "SASRec checkpoint vocab does not match the datamodule — "
    "the preprocessing above must reproduce the training-time item catalog."
)
sasrec = SASRec(SASRecConfig(**payload["model_config"]))
sasrec.load_state_dict(payload["model_state_dict"])
sasrec.eval()
print(f"SASRec loaded: {payload['model_config']}")

In [ ]:
# --- Sample random users with their prefix; full-catalog candidate set ---------
from flowrec.trajectories.semantic import SemanticTrajectoryBatch

N_USERS = 4
rng = np.random.default_rng(7)
test_idxs = rng.choice(len(dm.test_dataset), size=N_USERS, replace=False)
examples = [dm.test_dataset[int(i)] for i in test_idxs]
batch = dm._make_collator()(examples)

for col in range(N_USERS):
    h = int(batch["history_mask"][col].sum())
    prefix = batch["history_item_ids"][col][batch["history_mask"][col]].tolist()
    print(f"user {int(batch['user_ids'][col]):>6}  prefix({h}): {prefix}  ->  "
          f"target {int(batch['target_item_ids'][col])} "
          f"({batch['target_ratings'][col]:.0f}★, "
          f"{'verified' if bool(batch['target_verified_purchase'][col]) else 'unverified'})")

# Every catalog item is a candidate trajectory; the held-out target is positive.
catalog = torch.arange(1, dm.item_vocab_size, dtype=torch.long)  # 0 is padding
N_CATALOG = catalog.numel()
candidate_item_ids = catalog.unsqueeze(0).expand(N_USERS, N_CATALOG)
is_positive = candidate_item_ids.eq(batch["target_item_ids"].unsqueeze(1))
traj = SemanticTrajectoryBatch(
    candidate_token_ids=torch.zeros((N_USERS, N_CATALOG, 1), dtype=torch.long),
    candidate_token_mask=torch.ones((N_USERS, N_CATALOG, 1), dtype=torch.bool),
    is_positive=is_positive,
    extra={"candidate_item_ids": candidate_item_ids},
)
print(f"\ncandidates: {N_USERS} users x {N_CATALOG:,} catalog items")


In [ ]:
# --- Build all reward modes and score the full catalog per user ----------------
from flowrec.rewards.amazon_log import (
    AmazonLogRewardConfig,
    AmazonLogRewardModel,
    compute_item_mean_ratings,
)
from flowrec.rewards.mix import MultiplicativeRewardModel
from flowrec.rewards.sasrec import SASRecRewardConfig, SASRecRewardModel

# Additive mixing lives in the training script: MixedRewardModel is the exact
# model run_tiger builds for reward.name == "mix" (batch_max-normalized sum).
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from run_tiger import MixedRewardModel

item_means, global_mean = compute_item_mean_ratings(
    torch.tensor(dm.interactions["item_id"].to_numpy()),
    torch.tensor(dm.interactions["rating"].to_numpy(dtype="float32")),
    num_items=dm.item_vocab_size,
    shrinkage=10.0,
)
print(f"item mean ratings: global={global_mean:.3f}")

item_centered = AmazonLogRewardModel(
    AmazonLogRewardConfig(reward_mode="item_centered"),
    item_mean_rating=item_means,
    global_mean_rating=global_mean,
)

TRANSFORMS = ["softplus", "sigmoid", "identity", "exp", "minmax_batch", "rank_percentile"]
sasrec_by_transform = {
    t: SASRecRewardModel(sasrec, SASRecRewardConfig(transform=t)) for t in TRANSFORMS
}

REWARD_MODES = {
    "amazon_log / logged": AmazonLogRewardModel(AmazonLogRewardConfig()),
    "amazon_log / binary": AmazonLogRewardModel(AmazonLogRewardConfig(reward_mode="binary")),
    "amazon_log / item_centered": item_centered,
}
# SASRec under every transform, then its two derivatives: multiplication
# (MultiplicativeRewardModel) and summation (MixedRewardModel).
REWARD_MODES.update({f"sasrec / {t}": sasrec_by_transform[t] for t in TRANSFORMS})
REWARD_MODES.update({
    f"sasrec × ic / {t}": MultiplicativeRewardModel([sasrec_by_transform[t], item_centered])
    for t in TRANSFORMS
})
REWARD_MODES.update({
    f"sasrec + ic / {t}": MixedRewardModel([sasrec_by_transform[t], item_centered])
    for t in TRANSFORMS
})

rewards = {}
with torch.no_grad():
    for mode, model in REWARD_MODES.items():
        rewards[mode] = model(batch, traj).float()

# Where does the positive target sit inside each full-catalog distribution?
summary = pd.DataFrame(
    {
        mode: {
            f"user {int(batch['user_ids'][col])}": float(
                (r[col] < r[col][is_positive[col]]).float().mean() * 100
            )
            for col in range(N_USERS)
        }
        for mode, r in rewards.items()
    }
).T.round(1)
print("\nTarget percentile within the full-catalog reward distribution (higher = better separated):")
print(summary.to_string())

In [ ]:
# --- Plot: full-catalog reward distribution per user, one row per mode ---------
mode_names = list(REWARD_MODES)
fig, axes = plt.subplots(
    len(mode_names), N_USERS,
    figsize=(3.4 * N_USERS, 2.2 * len(mode_names)),
    sharex="row", sharey="row",
)

for row, mode in enumerate(mode_names):
    r = rewards[mode].numpy()
    # Rows whose reward spans >3 decades (exp, identity, the × ic products)
    # are binned and drawn on a log x-axis so the shape stays readable.
    lo, hi = max(r.min(), 1e-6), r.max()
    log_x = hi / lo > 1e3
    bins = (np.logspace(np.log10(lo), np.log10(hi * 1.001), 60)
            if log_x else np.linspace(lo, hi * 1.001, 60))
    for col in range(N_USERS):
        ax = axes[row, col]
        ax.hist(r[col], bins=bins, color=C_VERIFIED, edgecolor=SURFACE,
                linewidth=0.3, zorder=3)
        ax.set_yscale("log")
        if log_x:
            ax.set_xscale("log")
        tgt = float(r[col][is_positive[col].numpy()][0])
        pct = float((r[col] < tgt).mean() * 100)
        ax.axvline(tgt, color=C_UNVERIFIED, linewidth=1.6, zorder=4)
        ax.text(0.98, 0.92, f"target: {tgt:.3g}\npct {pct:.1f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=8, color=C_UNVERIFIED)
        ax.grid(axis="x", visible=False)
        if row == 0:
            h = int(batch["history_mask"][col].sum())
            rt = batch["target_ratings"][col].item()
            ver = "verified" if bool(batch["target_verified_purchase"][col]) else "unverified"
            ax.set_title(
                f"user {int(batch['user_ids'][col])} · prefix {h}\ntarget {rt:.0f}★ {ver}",
                fontsize=9, color=INK_2,
            )
        if col == 0:
            ax.set_ylabel(mode.replace(" / ", "\n"), fontsize=9, color=INK)

fig.suptitle(
    "Full-catalog reward distributions per user — amazon_log modes, sasrec under "
    "every transform, and its × / + derivatives with item_centered\n"
    f"({N_CATALOG:,} candidate items; orange line = positive target)",
    fontweight="bold", x=0.01, ha="left",
)
fig.tight_layout(rect=(0, 0, 1, 0.985))
plt.show()

### Reading the reward distributions

- **`logged` and `binary` are near-degenerate:** virtually the whole catalog sits
  at `unobserved_reward` (the tall spike at ~0), with only the target and a
  handful of history items separated. Within the positives these modes barely
  discriminate — and in `logged` mode a 1★ target collapses *below* the unobserved
  floor (percentile 0), i.e. the model is told the user's actual purchase is worse
  than a random unseen item.
- **`item_centered` is a multiplier, not a stand-alone reward:** the mass at 1.0
  is every unobserved item (neutral, "no evidence"), and the logged interactions
  spread across [0.25, 2] according to how far the rating sits from the item's
  typical rating (trust-weighted by the verified flag).
- **`sasrec / *` is dense:** a smooth, discriminating distribution over the whole
  catalog — but it knows nothing about the logged rating: it can rank a 1★
  purchase very highly (user 18739: pct 99.9). All transforms are monotone in
  the score, so the target percentile is **identical in every `sasrec / *` row**
  (78.5 / 82.5 / 5.4 / 99.9) — only the *shape* (the reward ratios) changes.
  The one exception is `identity`, which clamps negative scores to
  `reward_epsilon` and collapses roughly half the catalog (incl. user 24574's
  target) into a single spike.
- **`sasrec × ic` re-ranks with logged evidence while keeping the SASRec shape:**
  unobserved items keep their collaborative reward (multiplier 1.0), only logged
  items move — the 5★ targets push right (user 25878: 78.5 → 92.0 under
  softplus), the 1★ target pulls back (user 18739: 99.9 → 84.3 softplus,
  44.4 sigmoid). The unverified 5★ target (user 24574) barely moves
  (5.4 → 8.7) because the trust weight damps its multiplier.
- **`sasrec + ic` lets the modulator dominate:** `MixedRewardModel` divides each
  component by its batch max before summing, and the SASRec max is an outlier —
  so the normalized collaborative term is ≈ 0 for the bulk of the catalog and
  `item_centered` decides the ranking (user 24574's unverified 5★ jumps
  5.4 → 92.6). The distributions are visibly compressed (note the narrow x-range)
  and in the `exp` row the sum degenerates to `item_centered` alone: the 1★
  target falls back to percentile 0 — the cliff the multiplicative composition
  avoids.

### 5.1 Numeric summary — SASRec transforms and derivatives

The full-catalog plot above shows the shapes; the table below condenses every
`sasrec` variant (alone, `× ic`, `+ ic` — each under every
`SASRecRewardConfig.transform`) into three numbers:

- **target percentile** — where the positive target ranks in the full catalog.
  Pure transforms are monotone in the score, so it is identical across the
  `sasrec / *` rows; the `× ic` / `+ ic` derivatives *do* change it, because the
  logged-evidence modulator re-ranks observed items;
- **target/median contrast** — how strongly the reward prefers the target over a
  typical catalog item;
- **`log R` std** — the effective sharpness (temperature) of the reward
  landscape, which is what the trajectory-balance loss (`log R`) actually
  optimizes.

> **TB-safety note** (cf. `TB_SAFE_SASREC_TRANSFORMS` in
> `scripts/rewards_ablation.py`): `softplus`, `sigmoid`, `identity`, `exp` are
> pointwise — fixed terminal rewards — and so are their `× ic` products
> (`item_centered` is pointwise too). `minmax_batch` and `rank_percentile` are
> normalized *per candidate set*, and so is **every** `+ ic` row:
> `MixedRewardModel` divides each component by its batch max
> (`uses_candidate_set_normalization=True` in `scripts/run_tiger.py`). Those are
> inconsistent with a precomputed full-catalog logZ and shown for shape
> comparison only.

In [ ]:
# --- Numeric summary per sasrec variant (reuses the full-catalog rewards) ------
rows = []
for mode, r in rewards.items():
    if not mode.startswith("sasrec"):
        continue
    tgt = r[is_positive]
    pct = (r < tgt.unsqueeze(1)).float().mean(1) * 100
    contrast = tgt / r.median(dim=1).values
    log_r_std = torch.log(r.clamp_min(1e-6)).std()
    rows.append({
        "variant": mode,
        **{f"pct u{int(batch['user_ids'][i])}": round(float(p), 1) for i, p in enumerate(pct)},
        **{f"tgt/med u{int(batch['user_ids'][i])}": round(float(c), 2) for i, c in enumerate(contrast)},
        "logR std": round(float(log_r_std), 2),
    })
transform_summary = pd.DataFrame(rows)
print("Target percentile, target/median contrast, and logR std per sasrec variant:")
print(transform_summary.to_string(index=False))

### Reading the transform / derivative comparison

**Pure transforms (`sasrec / *`)** — percentiles identical in every row
(78.5 / 82.5 / 5.4 / 99.9); the transform only sets the contrast:

- **`sigmoid` compresses** (`log R` std ≈ 0.8): scores saturate toward 1, so the
  reward barely prefers the target over the median item (contrast ≈ 2–3×). Mild,
  stable, but low signal. **`minmax_batch` compresses even harder** (std ≈ 0.3).
- **`softplus` (default) is the balanced middle** (std ≈ 1.0): roughly
  log-normal, contrast 2–13×, no cliffs.
- **`exp` is the sharpest usable transform** (std ≈ 1.3, contrast up to ~200×):
  since TB works on `log R`, `exp` means the reward *is* the raw score up to
  temperature — sensible, but the right tail dominates logZ, so it effectively
  trains the policy to concentrate on top-scored items.
- **`rank_percentile`** is uniform by construction — pure ranking information
  with a bounded scale, but per-candidate-set (see TB-safety note above).
- **`identity`** clamps negative scores to `reward_epsilon`: half the catalog
  (and user 24574's target) collapses into the ε-spike, and its huge contrast
  numbers (~10⁵–10⁶) are meaningless ratios against that floor.

**Multiplication (`sasrec × ic`)** — keeps the base transform's `log R` std
*exactly* (the modulator is 1.0 almost everywhere) and re-ranks only logged
items: 5★ targets rise (u25878: 78.5 → 92.0 softplus, 98.8 sigmoid), the 1★
target drops (u18739: 99.9 → 84.3 softplus, 44.4 sigmoid), and the unverified
5★ moves barely at all (u24574: 5.4 → 8.7) because trust-weighting damps its
multiplier. How far the same multiplier moves an item depends on the local
density of the base distribution — flat `sigmoid` / `rank_percentile` translate
it into much larger rank shifts than heavy-tailed `exp` (u18739 stays at 98.8).
The u24574 jump under `identity` (0 → 80.2) is an artifact: any multiplier > 1
escapes the ε-spike. This is the composition `MultiplicativeRewardModel` was
designed for.

**Summation (`sasrec + ic`)** — batch_max normalization makes the sum
modulator-dominated: SASRec's max is an outlier, so its normalized term is ≈ 0
for the bulk of the catalog and `item_centered` decides the ranking (u24574's
unverified 5★ leaps 5.4 → 92.6, ignoring the trust damping that the product
respects). `log R` std collapses to 0.01–0.31 (vs ≈ 1 for the base transforms) —
a very weak TB gradient — and with `exp` the collaborative term vanishes
entirely: the row degenerates to `item_centered` alone and the 1★ target lands
at percentile 0, the exact `logged`-mode cliff the multiplicative composition
avoids.

### 5.2 Semantic-ID similarity to the positive item

Before analyzing similarity-based rewards, we look at the raw similarity signal
itself. `SimilarityRewardModel` (`flowrec/rewards/similarity.py`) defines the
similarity of a candidate to the positive target as the **number of semantic-ID
token positions they share, scaled to [0, 1]** — positionwise equality, not
prefix overlap. The semantic IDs are the RQ codes from
`datasets/semantic_ids/exp1-standard params/instruments/semantic_ids.json` —
the same file the datamodule loads in the setup cell. Each item has **4 RQ
levels from 256-entry codebooks plus a 5th collision-disambiguation token**
(appended by the tokenizer's `random_append` policy to separate the 118 items
whose 4-level codes collided); tokens are offset by one so 0 stays reserved
for padding.

For each of the four users we compute the similarity of **every catalog item**
to the held-out positive and look at its distribution. With `sid_len = 5` the
similarity can only take six values: 0, 0.2, …, 1. The cell below also rebuilds
the full-catalog candidate batch with the *real* semantic tokens (`traj_sem`) —
unlike `traj` above, whose dummy tokens only support item-level reward models —
so token-level reward models can score it directly.

In [ ]:
# --- Semantic-ID similarity of every catalog item to the positive target -------
from flowrec.rewards.similarity import SimilarityRewardConfig, SimilarityRewardModel

SID_LEN = dm.sid_len
print(f"semantic IDs: {dm.semantic_id_path}")
print(f"sid_len={SID_LEN}, items with semantic IDs: {len(dm.semantic_ids):,}")

# Semantic-ID token table for the whole catalog (row = internal item id).
# dm.semantic_ids is keyed by raw ASIN with tokens already offset by one
# (0 is reserved for padding), and covers every catalog item because the
# datamodule dropped items without a semantic ID.
token_ids_by_item = torch.zeros((dm.item_vocab_size, SID_LEN), dtype=torch.long)
token_mask_by_item = torch.zeros((dm.item_vocab_size, SID_LEN), dtype=torch.bool)
for item_id in range(1, dm.item_vocab_size):
    tokens = dm.semantic_ids[dm.id_to_raw_item(item_id)]
    n = min(len(tokens), SID_LEN)
    token_ids_by_item[item_id, :n] = torch.as_tensor(tokens[:n], dtype=torch.long)
    token_mask_by_item[item_id, :n] = True
assert bool(token_mask_by_item[1:].all()), "every catalog item should have a full semantic ID"

# Same full-catalog candidate set as above, but with REAL semantic tokens, so
# token-level reward models (SimilarityRewardModel) can score it too.
traj_sem = SemanticTrajectoryBatch(
    candidate_token_ids=token_ids_by_item[catalog].unsqueeze(0).expand(N_USERS, N_CATALOG, SID_LEN),
    candidate_token_mask=token_mask_by_item[catalog].unsqueeze(0).expand(N_USERS, N_CATALOG, SID_LEN),
    is_positive=is_positive,
    extra={"candidate_item_ids": candidate_item_ids},
)

# sanity: the collator's target tokens must equal the catalog rows of the targets
assert torch.equal(batch["target_token_ids"], token_ids_by_item[batch["target_item_ids"]])

similarity_model = SimilarityRewardModel(SimilarityRewardConfig())
with torch.no_grad():
    similarity = similarity_model(batch, traj_sem).float()  # [B, N] = matched positions / SID_LEN
matched = (similarity * SID_LEN).round().long()             # 0..SID_LEN matching token positions

# Distribution of similarity levels per user.
levels = torch.arange(SID_LEN + 1)
dist = pd.DataFrame(
    {
        f"user {int(batch['user_ids'][col])}": [
            int((matched[col] == k).sum()) for k in levels
        ]
        for col in range(N_USERS)
    },
    index=[f"{int(k)}/{SID_LEN} (sim {k / SID_LEN:.2f})" for k in levels],
)
dist.loc["mean similarity"] = [round(float(similarity[c].mean()), 4) for c in range(N_USERS)]
print("\nItems by number of semantic-ID positions shared with the positive target:")
print(dist.to_string())

# How discriminative is each codebook level? Items sharing the target's token
# at position l (independently of the other positions).
n_codes = int(token_ids_by_item[1:].max())
per_pos = pd.DataFrame(
    {
        f"user {int(batch['user_ids'][col])}": [
            int((token_ids_by_item[catalog][:, l] == batch["target_token_ids"][col, l]).sum())
            for l in range(SID_LEN)
        ]
        for col in range(N_USERS)
    },
    index=[f"position {l + 1}" for l in range(SID_LEN)],
)
print(f"\nItems sharing the target's token at each position (of {N_CATALOG:,}; chance ≈ {N_CATALOG / n_codes:.0f}):")
print(per_pos.to_string())

In [ ]:
# --- Plot: distribution of semantic-ID similarity to the positive target -------
fig, axes = plt.subplots(1, N_USERS, figsize=(3.4 * N_USERS, 3.2), sharey=True)

level_axis = np.arange(SID_LEN + 1)
for col in range(N_USERS):
    ax = axes[col]
    counts = np.array([int((matched[col] == k).sum()) for k in level_axis])
    colors = [C_VERIFIED] * SID_LEN + [C_UNVERIFIED]  # 4/4 is the target itself
    ax.bar(level_axis, counts, width=0.7, color=colors, edgecolor=SURFACE,
           linewidth=1.2, zorder=3)
    ax.set_yscale("log")
    ax.set_ylim(0.7, counts.max() * 4)
    for x, c in zip(level_axis, counts):
        if c > 0:
            ax.text(x, c, f"{c:,}", ha="center", va="bottom",
                    fontsize=8.5, color=INK_2)
    ax.set_xticks(level_axis)
    ax.set_xticklabels([f"{k}/{SID_LEN}" for k in level_axis])
    ax.set_xlabel("shared semantic-ID positions")
    ax.grid(axis="x", visible=False)
    rt = batch["target_ratings"][col].item()
    ver = "verified" if bool(batch["target_verified_purchase"][col]) else "unverified"
    ax.set_title(f"user {int(batch['user_ids'][col])} · target {rt:.0f}★ {ver}",
                 fontsize=9, color=INK_2)
axes[0].set_ylabel("catalog items (log)")

fig.suptitle(
    "Semantic-ID similarity to the positive target across the full catalog "
    f"(similarity = shared positions / {SID_LEN}; orange = the target itself)",
    fontweight="bold", x=0.01, ha="left",
)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

### Reading the similarity distribution

- **The landscape is a delta at the target plus thin noise.** 97–99% of the
  catalog shares **0 of 5** positions with the target (10,135–10,353 of 10,439
  items), 0.8–2.9% shares exactly one (85–304 items), at most 5 items share
  two, **no item shares three or more**, and the only 5/5 item is the target
  itself. Mean similarity is 0.002–0.006.
- **A single shared position is near-chance information.** Each codebook has
  256 entries, so a random item matches a given position with p ≈ 1/256 —
  about 41 of 10,439 items. The observed per-position counts (10–96) scatter
  around that level, i.e. the RQ levels are individually weak discriminators
  and roughly independent; sharing one token says little about sharing another.
  (User 18739's counts of 10–20 across positions mean its target sits in rare
  clusters at every level.)
- **The measure is positionwise, not hierarchical.** `SimilarityRewardModel`
  counts a match at any position equally, so an item agreeing only in the last
  token (the random collision suffix!) scores the same 0.2 as one agreeing in
  the first (coarse cluster) — the RQ hierarchy is ignored.
- **Consequence for a similarity-based reward:** with only 6 possible values
  and ~98% of the mass at the ε-floor, the terminal reward is close to
  `amazon_log / binary` — a spike on the target with a small graded shoulder of
  a few dozen "cousins" — rather than a dense, smooth signal like `sasrec`.

### 5.3 Gower over semantic-ID tokens — ordinal within-codebook distance

The exact-match token similarity above is near-binary, and direct token matches
are rare. Here we reformulate the Gower logic **using semantic IDs only** (no
ratings, popularity, or categories): each codebook level becomes one Gower
feature, and levels get **different weights** following the RQ residual
hierarchy (level 1 encodes the coarsest structure, deeper levels only refine
it).

**Weighted Gower.** For items $a, b$ with codes $c_{a\ell}$ at levels
$\ell = 1 \dots L$ (every level is observed for every item, so Gower's
missing-feature indicator $\delta_\ell$ is always 1 and drops out):

$$
S(a,b) = \frac{\sum_{\ell=1}^{L} w_\ell\, s_\ell(a,b)}{\sum_{\ell=1}^{L} w_\ell},
\qquad
D_{\text{Gower}}(a,b) = 1 - S(a,b),
\qquad
w_\ell = 2^{-(\ell-1)} = 1, \tfrac{1}{2}, \tfrac{1}{4}, \dots
$$

The terminal reward against the positive target $t$ is
$R(a) = \max\bigl(S(a,t),\, \varepsilon\bigr)$ with $\varepsilon = 10^{-6}$.
The per-level similarity $s_\ell \in [0,1]$ is what distinguishes the variants:

- **exact match** (categorical Gower):
  $s_\ell = \mathbb{1}\left[c_{a\ell} = c_{b\ell}\right]$
- **index distance** (numeric Gower on token indices; assumes nearby indices —
  50 vs 51 — encode similar things):
  $s_\ell = 1 - \dfrac{\lvert c_{a\ell} - c_{b\ell} \rvert}{R_\ell}$,
  with $R_\ell \approx 255$ the index range at level $\ell$
- **codebook distance** (numeric Gower in the quantizer's latent space):
  $s_\ell = 1 - \dfrac{\lVert \mathbf{e}_\ell(c_{a\ell}) - \mathbf{e}_\ell(c_{b\ell}) \rVert_2}{\max_{u,v} \lVert \mathbf{e}_\ell(u) - \mathbf{e}_\ell(v) \rVert_2}$,
  where $\mathbf{e}_\ell(\cdot)$ is the RQ-VAE codebook vector at level $\ell$
  (loaded from `model.pt` next to the semantic IDs in
  `datasets/semantic_ids/exp1-standard params/instruments/`); the max-distance
  denominator is Gower's feature range carried into a metric space

| variant | per-level similarity $s_\ell$ | level weights $w_\ell$ |
|---|---|---|
| `exact match · equal w` | exact match | equal — reduces *exactly* to the 5.2 similarity (asserted) |
| `\|Δtoken\| · equal w` | index distance | equal |
| `\|Δtoken\| · level-decay w` | index distance | $2^{-(\ell-1)}$: 1, ½, ¼, … |
| `codebook dist · equal w` | codebook distance | equal |
| `codebook dist · level-decay w` | codebook distance | $2^{-(\ell-1)}$: 1, ½, ¼, … |

The 5th token is the collision-disambiguation suffix (`random_append`) — it has
no codebook behind it, so every mode treats it as exact match. Both distance
assumptions are **tested** in the cell after the plot, using the tuned SASRec
item embeddings as a similarity oracle.

In [ ]:
# --- Gower over semantic-ID tokens: index distance vs codebook-vector distance --
# Three per-level similarity treatments, combined by the same weighted-Gower
# formula (weights decay 1, 1/2, 1/4, ... down the RQ hierarchy):
#   exact    - 1{token equal}                     (categorical; the 5.2 measure)
#   index    - 1 - |dtoken| / range               (assumes nearby indices ~ similar)
#   codebook - 1 - ||c_a - c_b|| / max_pair_dist  (actual RQ-VAE codebook geometry)
# The exp1 IDs have 4 RQ levels + 1 collision-disambiguation token appended by
# collision_policy="random_append"; that 5th token has no codebook and is
# treated as exact-match in every mode.

RQVAE_DIR = PROJECT_ROOT / "datasets/semantic_ids/exp1-standard params/instruments"
assert Path(dm.semantic_id_path) == RQVAE_DIR / "semantic_ids.json", (
    "The RQ-VAE codebooks must come from the tokenizer that produced the "
    "datamodule's semantic IDs."
)
codebooks = torch.load(RQVAE_DIR / "model.pt", map_location="cpu", weights_only=False)[
    "model_state_dict"
]["quantizer.codebooks"].float()                                 # [L_rq, K, D]
N_RQ_LEVELS, K_CODES, LATENT_DIM = codebooks.shape
print(f"RQ-VAE codebooks: {N_RQ_LEVELS} levels x {K_CODES} codes x {LATENT_DIM}d "
      f"(sid_len={SID_LEN} -> {SID_LEN - N_RQ_LEVELS} collision-suffix level(s))")

# Pairwise codebook-vector distances per level, normalized by the level's max.
code_dist = torch.cdist(codebooks, codebooks)                    # [L_rq, K, K]
code_dist = (code_dist / code_dist.amax(dim=(1, 2), keepdim=True)).numpy()

TOKEN_OFFSET = 1 if dm.cfg.offset_semantic_ids_by_one else 0
raw_codes = token_ids_by_item.numpy() - TOKEN_OFFSET             # [V, L], 0-based
assert raw_codes[1:, :N_RQ_LEVELS].min() >= 0 and raw_codes[1:, :N_RQ_LEVELS].max() < K_CODES
tok_ranges = raw_codes[1:].max(axis=0) - raw_codes[1:].min(axis=0)
LEVEL_WEIGHTS = 2.0 ** -np.arange(SID_LEN)                       # [1, .5, .25, ...]


def token_gower(weights, mode):
    """Weighted Gower similarity to each user's target over token levels.

    mode: 'exact' | 'index' | 'codebook'. The collision-suffix levels (beyond
    the RQ codebooks) always use exact match.
    """
    w = np.asarray(weights, dtype=np.float64)
    sims = np.zeros((N_USERS, N_CATALOG))
    for col in range(N_USERS):
        t = int(batch["target_item_ids"][col])
        s = np.empty((N_CATALOG, SID_LEN))
        for level in range(SID_LEN):
            ci, ct = raw_codes[catalog, level], raw_codes[t, level]
            if mode == "exact" or level >= N_RQ_LEVELS:
                s[:, level] = ci == ct
            elif mode == "index":
                s[:, level] = 1.0 - np.abs(ci - ct) / tok_ranges[level]
            else:
                s[:, level] = 1.0 - code_dist[level, ci, ct]
        sims[col] = (s * w).sum(axis=1) / w.sum()
    return torch.from_numpy(sims).float()


TOKEN_GOWER_VARIANTS = {
    "exact match · equal w": dict(weights=np.ones(SID_LEN), mode="exact"),
    "|Δtoken| · equal w": dict(weights=np.ones(SID_LEN), mode="index"),
    "|Δtoken| · level-decay w": dict(weights=LEVEL_WEIGHTS, mode="index"),
    "codebook dist · equal w": dict(weights=np.ones(SID_LEN), mode="codebook"),
    "codebook dist · level-decay w": dict(weights=LEVEL_WEIGHTS, mode="codebook"),
}
gower_rewards = {
    name: token_gower(**kw).clamp_min(1e-6) for name, kw in TOKEN_GOWER_VARIANTS.items()
}

# sanity: the exact-match variant is the 5.2 positionwise similarity
assert float((gower_rewards["exact match · equal w"] - similarity).abs().max()) < 1e-5

rows = []
for name, r in gower_rewards.items():
    tgt = r[is_positive]
    pct = (r < tgt.unsqueeze(1)).float().mean(1) * 100
    contrast = tgt / r.median(dim=1).values
    log_r_std = torch.log(r.clamp_min(1e-6)).std()
    rows.append({
        "variant": name,
        **{f"pct u{int(batch['user_ids'][i])}": round(float(p), 1) for i, p in enumerate(pct)},
        **{f"tgt/med u{int(batch['user_ids'][i])}": round(float(c), 2) for i, c in enumerate(contrast)},
        "logR std": round(float(log_r_std), 2),
    })
print("\nToken-Gower reward: target percentile, target/median contrast, logR std:")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# --- Plot: reward = token-level Gower similarity, full catalog per user --------
variant_names = list(gower_rewards)
fig, axes = plt.subplots(
    len(variant_names), N_USERS,
    figsize=(3.4 * N_USERS, 2.4 * len(variant_names)),
    sharex=True, sharey="row",
)
bins = np.linspace(0, 1.001, 60)
for row, name in enumerate(variant_names):
    r = gower_rewards[name].numpy()
    for col in range(N_USERS):
        ax = axes[row, col]
        ax.hist(r[col], bins=bins, color=C_VERIFIED, edgecolor=SURFACE,
                linewidth=0.3, zorder=3)
        ax.set_yscale("log")
        tgt = float(r[col][is_positive[col].numpy()][0])
        pct = float((r[col] < tgt).mean() * 100)
        ax.axvline(tgt, color=C_UNVERIFIED, linewidth=1.6, zorder=4)
        ax.text(0.02, 0.92, f"target: {tgt:.3g}\npct {pct:.1f}",
                transform=ax.transAxes, ha="left", va="top",
                fontsize=8, color=C_UNVERIFIED)
        ax.grid(axis="x", visible=False)
        if row == 0:
            rt = batch["target_ratings"][col].item()
            ver = "verified" if bool(batch["target_verified_purchase"][col]) else "unverified"
            ax.set_title(f"user {int(batch['user_ids'][col])} · target {rt:.0f}★ {ver}",
                         fontsize=9, color=INK_2)
        if col == 0:
            ax.set_ylabel(name.replace(" · ", "\n"), fontsize=9, color=INK)

fig.suptitle(
    "Reward = token-level Gower similarity to the positive target "
    "(ordinal |Δtoken| within each codebook level; orange line = target)",
    fontweight="bold", x=0.01, ha="left",
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

In [ ]:
# --- Which distance carries signal: |Δtoken| or codebook geometry? -------------
# Proxy check with the tuned SASRec item embeddings: mean cosine similarity of
# random item pairs, grouped (per RQ level) by (a) token index distance and
# (b) normalized codebook-vector distance. If a distance is meaningful, cosine
# should DECAY as it grows; a flat profile means the distance carries no
# information beyond the exact match.
E = torch.nn.functional.normalize(sasrec.item_embedding.weight.detach()[catalog], dim=1)
pair_rng = np.random.default_rng(0)
pairs = pair_rng.integers(0, N_CATALOG, size=(400_000, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]
cos = (E[pairs[:, 0]] * E[pairs[:, 1]]).sum(dim=1).numpy()
codes_cat = raw_codes[catalog]

IDX_BINS = [("|Δ| = 0", 0, 0), ("|Δ| = 1", 1, 1), ("|Δ| 2-5", 2, 5),
            ("|Δ| 6-20", 6, 20), ("|Δ| > 20", 21, 10**9)]
idx_rows, cb_rows = [], []
for level in range(N_RQ_LEVELS):
    a, b = codes_cat[pairs[:, 0], level], codes_cat[pairs[:, 1], level]
    d_idx = np.abs(a - b)
    idx_rows.append({
        "level": level + 1,
        **{lab: round(float(cos[(d_idx >= lo) & (d_idx <= hi)].mean()), 4)
           for lab, lo, hi in IDX_BINS},
    })
    d_cb = code_dist[level, a, b]
    nz = d_cb[d_cb > 0]
    qs = np.quantile(nz, [0.25, 0.5, 0.75])
    cb_bins = [("dist = 0", d_cb == 0), ("Q1 (closest)", (d_cb > 0) & (d_cb <= qs[0])),
               ("Q2", (d_cb > qs[0]) & (d_cb <= qs[1])), ("Q3", (d_cb > qs[1]) & (d_cb <= qs[2])),
               ("Q4 (farthest)", d_cb > qs[2])]
    cb_rows.append({
        "level": level + 1,
        **{lab: round(float(cos[m].mean()), 4) for lab, m in cb_bins},
    })
print(f"Mean SASRec-embedding cosine of item pairs (baseline = {cos.mean():.4f})")
print("\nby token INDEX distance:")
print(pd.DataFrame(idx_rows).to_string(index=False))
print("\nby CODEBOOK-vector distance (quartiles of nonzero distances):")
print(pd.DataFrame(cb_rows).to_string(index=False))

### Reading the token-Gower reward — index distance vs codebook geometry

- **All ordinal variants fix the sparsity, but are flat as rewards.** The delta
  landscape becomes a dense distribution, the target keeps similarity 1.0 and
  percentile 100 (by construction). Target/median contrast is ≈ 1.4–2.1 and
  `log R` std 0.11–0.24 (vs 1.79 for exact match) — differences average out
  across levels.
- **The index-distance assumption is refuted.** Sharing the *level-1* token is
  real signal (pair cosine 0.134 ≈ 2.4× the 0.055 random baseline), but pairs
  whose level-1 tokens differ by **just 1** sit at the baseline (0.052), and the
  profile is flat across every |Δ| > 0 bin. Codebook indices are nominal labels
  — index 51 is unrelated to index 50 — so the graded shoulder of the `|Δtoken|`
  rows is noise.
- **Codebook geometry, by contrast, is genuinely ordinal at level 1.** Mean pair
  cosine decays *monotonically* with codebook-vector distance:
  `0.146 (same code) → 0.075 (closest quartile) → 0.059 → 0.049 → 0.035
  (farthest)`. Pairs in the closest quartile are ~36% more similar than random,
  the farthest ~36% less — exactly the ordering the Gower numeric treatment
  needs. This is the option that works.
- **Levels 2–4 carry almost no item-level signal** — even at distance 0 their
  cosine (0.053–0.060) barely rises above the baseline: the residual levels
  refine reconstruction, not item identity. That is a strong empirical
  justification for the decaying level weights: effectively all usable
  similarity lives in level 1.
- **Verdict:** `codebook dist · level-decay w` is the defensible token-only
  similarity reward — graded by the quantizer's true latent geometry and
  dominated by the one level that carries signal. Caveats as before: it is
  target-conditioned (percentile 100 by construction) and mild
  (`log R` std ≈ 0.14), so if used as a TB terminal reward it likely needs
  sharpening (a power/exponential on the similarity) to give the loss a usable
  gradient; the collision-suffix level stays exact-match (it is a random
  tie-breaker with no geometry behind it).

### 5.4 Continuous semantic-embedding similarity — skipping the quantizer

The semantic-ID tokens are a quantization of 384-d content embeddings
(`all-MiniLM-L6-v2` over the item metadata text —
`datasets/embeddings/exp1-standard params/instruments_content_embeddings.pt`,
the exact input the exp1 RQ-VAE was trained on). Section 5.3 measured
similarity *through* the quantizer (token matches, codebook vectors); here we
drop the quantizer entirely and measure distance between the items' embeddings
themselves:

- **embedding L2** — the same Gower recipe as before, applied to one continuous
  feature: $s(a,t) = 1 - \dfrac{\lVert z_a - z_t \rVert_2}{D_{\max}}$, with
  $D_{\max}$ the exact catalog diameter (max pairwise distance);
- **embedding cosine** — $s(a,t) = \dfrac{\cos(z_a, z_t) + 1}{2}$, insensitive
  to vector norms.

The two are **not** equivalent here: the MiniLM embeddings are unnormalized
(norms 1.3–5.2), so L2 mixes direction with norm — and the norm tracks text
length more than meaning. The check cell after the plot bins random item pairs
by both distances against the SASRec-embedding oracle to see which one
actually orders behavioral similarity.

In [ ]:
# --- Similarity in the raw content-embedding space (no quantization) -----------
EMB_PATH = PROJECT_ROOT / "datasets/embeddings/exp1-standard params/instruments_content_embeddings.pt"
emb_payload = torch.load(EMB_PATH, map_location="cpu", weights_only=False)
emb_ids = [str(x) for x in emb_payload["item_ids"]]
emb_matrix = emb_payload["embeddings"].float()
print(f"content embeddings: {tuple(emb_matrix.shape)}, embedder: {emb_payload['metadata']['embedder']}")

# Reindex to catalog order (row = internal item id - 1).
row_by_raw = {raw: i for i, raw in enumerate(emb_ids)}
emb_rows = [row_by_raw[dm.id_to_raw_item(i)] for i in range(1, dm.item_vocab_size)]
Z = emb_matrix[torch.tensor(emb_rows)]                       # [N_CATALOG, 384]
Zn = torch.nn.functional.normalize(Z, dim=1)
norms = Z.norm(dim=1)
print(f"embedding norms: min={norms.min():.2f} med={norms.median():.2f} "
      f"max={norms.max():.2f} (unnormalized -> L2 and cosine differ)")

# Exact catalog diameter = Gower's "feature range" in this metric space.
D_max = max(float(torch.cdist(chunk, Z).max()) for chunk in torch.split(Z, 1024))
print(f"catalog diameter D_max = {D_max:.3f}")

emb_rewards = {
    "embedding L2 · 1−‖Δz‖/Dmax": torch.zeros((N_USERS, N_CATALOG)),
    "embedding cosine · (cos+1)/2": torch.zeros((N_USERS, N_CATALOG)),
}
for col in range(N_USERS):
    t = int(batch["target_item_ids"][col]) - 1               # catalog row of the target
    emb_rewards["embedding L2 · 1−‖Δz‖/Dmax"][col] = (
        1.0 - (Z - Z[t]).norm(dim=1) / D_max
    ).clamp_min(1e-6)
    emb_rewards["embedding cosine · (cos+1)/2"][col] = (
        (Zn @ Zn[t] + 1.0) / 2.0
    ).clamp_min(1e-6)

rows = []
for name, r in emb_rewards.items():
    tgt = r[is_positive]
    pct = (r < tgt.unsqueeze(1)).float().mean(1) * 100
    contrast = tgt / r.median(dim=1).values
    log_r_std = torch.log(r.clamp_min(1e-6)).std()
    rows.append({
        "variant": name,
        **{f"pct u{int(batch['user_ids'][i])}": round(float(p), 1) for i, p in enumerate(pct)},
        **{f"tgt/med u{int(batch['user_ids'][i])}": round(float(c), 2) for i, c in enumerate(contrast)},
        "logR std": round(float(log_r_std), 2),
    })
print("\nEmbedding-similarity reward: target percentile, target/median contrast, logR std:")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# --- Plot: reward = content-embedding similarity, full catalog per user --------
emb_variant_names = list(emb_rewards)
fig, axes = plt.subplots(
    len(emb_variant_names), N_USERS,
    figsize=(3.4 * N_USERS, 2.4 * len(emb_variant_names)),
    sharex="row", sharey="row",
)
for row, name in enumerate(emb_variant_names):
    r_all = emb_rewards[name].numpy()
    bins = np.linspace(r_all.min(), max(r_all.max(), 1.0) * 1.001, 60)
    for col in range(N_USERS):
        ax = axes[row, col]
        r = r_all[col]
        ax.hist(r, bins=bins, color=C_VERIFIED, edgecolor=SURFACE,
                linewidth=0.3, zorder=3)
        ax.set_yscale("log")
        tgt = float(r[is_positive[col].numpy()][0])
        pct = float((r < tgt).mean() * 100)
        ax.axvline(tgt, color=C_UNVERIFIED, linewidth=1.6, zorder=4)
        ax.text(0.02, 0.92, f"target: {tgt:.3g}\npct {pct:.1f}",
                transform=ax.transAxes, ha="left", va="top",
                fontsize=8, color=C_UNVERIFIED)
        ax.grid(axis="x", visible=False)
        if row == 0:
            rt = batch["target_ratings"][col].item()
            ver = "verified" if bool(batch["target_verified_purchase"][col]) else "unverified"
            ax.set_title(f"user {int(batch['user_ids'][col])} · target {rt:.0f}★ {ver}",
                         fontsize=9, color=INK_2)
        if col == 0:
            ax.set_ylabel(name.replace(" · ", "\n"), fontsize=9, color=INK)

fig.suptitle(
    "Reward = content-embedding similarity to the positive target "
    "(no quantization; orange line = target)",
    fontweight="bold", x=0.01, ha="left",
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

In [ ]:
# --- Which embedding distance orders behavioral similarity: L2 or cosine? ------
# Same oracle as in 5.3: mean SASRec-embedding cosine of random item pairs,
# binned by the content-embedding distance between the pair's items.
E = torch.nn.functional.normalize(sasrec.item_embedding.weight.detach()[catalog], dim=1)
pair_rng = np.random.default_rng(0)
pairs = pair_rng.integers(0, N_CATALOG, size=(400_000, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]
cos_beh = (E[pairs[:, 0]] * E[pairs[:, 1]]).sum(dim=1).numpy()

d_l2 = (Z[pairs[:, 0]] - Z[pairs[:, 1]]).norm(dim=1).numpy()
cos_content = (Zn[pairs[:, 0]] * Zn[pairs[:, 1]]).sum(dim=1).numpy()

def binned(metric, closer_is_smaller):
    """Mean behavioral cosine in percentile bins of `metric`, closest first."""
    pcts = [1, 25, 50, 75] if closer_is_smaller else [99, 75, 50, 25]
    e = np.percentile(metric, pcts)
    if closer_is_smaller:
        masks = [("closest 1%", metric <= e[0]), ("1-25%", (metric > e[0]) & (metric <= e[1])),
                 ("25-50%", (metric > e[1]) & (metric <= e[2])), ("50-75%", (metric > e[2]) & (metric <= e[3])),
                 ("farthest 25%", metric > e[3])]
    else:
        masks = [("closest 1%", metric >= e[0]), ("1-25%", (metric < e[0]) & (metric >= e[1])),
                 ("25-50%", (metric < e[1]) & (metric >= e[2])), ("50-75%", (metric < e[2]) & (metric >= e[3])),
                 ("farthest 25%", metric < e[3])]
    return {lab: round(float(cos_beh[m].mean()), 4) for lab, m in masks}

from scipy.stats import spearmanr
emb_check = pd.DataFrame([
    {"distance": "L2 ‖Δz‖ (raw)", **binned(d_l2, closer_is_smaller=True)},
    {"distance": "cosine (normalized)", **binned(cos_content, closer_is_smaller=False)},
])
print(f"Mean SASRec-embedding cosine by content-embedding proximity (baseline = {cos_beh.mean():.4f}):")
print(emb_check.to_string(index=False))
print(f"\nspearman(content cosine, behavioral cosine) over 100k pairs: "
      f"{spearmanr(cos_content[:100_000], cos_beh[:100_000]).statistic:.3f}")
print("5.3 reference: same-level-1-code pairs average 0.146 - the quantizer's "
      "coarse partition is MORE behaviorally coherent than raw top-1% cosine neighborhoods.")

### Reading the embedding-similarity reward

- **Both variants are dense** and, as with every similarity-to-target reward,
  crown the target (percentile 100 by construction). `embedding L2` gives
  contrast 1.8–3.0 and `log R` std 0.24; `embedding cosine` is flatter
  (contrast ≈ 1.55, `log R` std 0.08) because $(\cos + 1)/2$ compresses the
  catalog into roughly [0.3, 1].
- **The oracle check picks cosine over L2 decisively.** Binned by content
  *cosine*, behavioral similarity decays monotonically through every bin
  (0.107 → 0.069 → 0.054 → 0.049 → 0.047 against the 0.055 baseline) — a
  genuine, if soft, global ordering. Binned by raw *L2*, the profile is flat
  beyond the closest 1% (0.081 → 0.058 → ≈ baseline): the norm component of
  the unnormalized MiniLM vectors is noise. If this reward is used, use the
  cosine form (equivalently, L2 on normalized embeddings).
- **The quantizer is not just a lossy copy of this space — it sharpens it.**
  Pairs sharing the level-1 RQ code (≈ 0.6% of pairs) average behavioral
  cosine **0.146**, clearly above the **0.107** of the top-1% content-cosine
  pairs. The RQ-VAE's learned encoder + partition concentrates behaviorally
  similar items better than raw text-embedding neighborhoods do. Beyond the
  near neighborhood, though, content cosine keeps grading (0.069 in the
  1–25% bin) where codebook levels 2–4 were silent.
- **Content similarity is a weak global predictor of behavior:**
  spearman(content cosine, behavioral cosine) ≈ 0.08 over random pairs —
  nearly all of its information lives in the top quartile, and below the
  median "far in text space" means almost nothing behaviorally.
- **The similarity ladder so far:** exact tokens (delta, strong but binary) →
  codebook geometry (graded at level 1) → embedding cosine (graded globally,
  weak) — each step denser and softer. All are target-conditioned; for a TB
  reward the natural next step is multiplying one of these graded similarities
  with a collaborative signal (`sasrec × similarity`), mirroring the `× ic`
  composition of section 5.

### 5.5 Hierarchical analysis — conditioning on the prefix

The checks in 5.3 treated every level **independently**, but RQ levels are
residuals: the level-2 code describes the residual *within* a level-1 cluster,
so the same level-2 token in two different level-1 clusters refers to two
unrelated residual directions. A marginal (per-level) analysis therefore
dilutes any real signal that deeper levels carry. This section redoes the
analysis **conditionally**:

1. **Exact-prefix-depth ladder.** Pairs are grouped by the *exact* depth of
   their shared prefix (levels matching contiguously from level 1). Random
   sampling cannot populate the deep bins — sharing a 2-prefix is a ~10⁻⁵
   event — so all within-cluster pairs are enumerated directly. If the
   hierarchy is real, behavioral coherence must climb with depth.
2. **The marginal confound, made explicit.** Pairs that share a level-2 (or
   level-3) token *without* sharing level 1 should sit at the random baseline —
   that is what buried the deep levels in 5.3.
3. **Conditional geometry.** Within same-level-1 pairs: does the *level-2*
   codebook distance grade similarity in its proper context?
4. **A prefix-gated reward.** The Gower formula rebuilt to respect the
   hierarchy: level $\ell$ contributes only while the prefix above it matches;
   the first divergent level contributes its codebook-geometry similarity;
   everything deeper contributes 0:

$$
S_{\text{hier}}(a,b) = \frac{1}{\sum_\ell w_\ell} \sum_{\ell=1}^{L} w_\ell \,
\underbrace{\mathbb{1}\!\left[c_{a,1..\ell-1} = c_{b,1..\ell-1}\right]}_{\text{prefix gate}}
\cdot g_\ell(a,b), \qquad
g_\ell = \begin{cases}
1 & c_{a\ell} = c_{b\ell} \\
1 - \dfrac{\lVert \mathbf{e}_\ell(c_{a\ell}) - \mathbf{e}_\ell(c_{b\ell}) \rVert}{\max\lVert\cdot\rVert} & \text{first divergence}
\end{cases}
$$

(the collision-suffix level stays exact-match). Both weight settings are shown —
under prefix gating the argument for decaying weights changes, as the check
below makes clear.

In [ ]:
# --- Prefix-depth analysis: enumerate within-cluster pairs, condition on prefix -
E = torch.nn.functional.normalize(sasrec.item_embedding.weight.detach()[catalog], dim=1).numpy()
codes_cat = raw_codes[catalog.numpy()]                        # [N, SID_LEN]
rq = codes_cat[:, :N_RQ_LEVELS]

# All pairs sharing the level-1 code (deep-prefix bins are ~1e-5 events under
# random sampling, so enumerate instead).
groups = {}
for i, c1 in enumerate(rq[:, 0]):
    groups.setdefault(int(c1), []).append(i)
pair_a, pair_b = [], []
for idxs in groups.values():
    if len(idxs) < 2:
        continue
    a = np.array(idxs)
    ii, jj = np.triu_indices(len(a), k=1)
    pair_a.append(a[ii]); pair_b.append(a[jj])
pair_a = np.concatenate(pair_a); pair_b = np.concatenate(pair_b)
print(f"within-level-1-cluster pairs: {len(pair_a):,} "
      f"({len(groups)} clusters, largest {max(len(v) for v in groups.values())})")

cos_in = (E[pair_a] * E[pair_b]).sum(axis=1)
eq2 = rq[pair_a, 1] == rq[pair_b, 1]
eq3 = rq[pair_a, 2] == rq[pair_b, 2]
eq4 = rq[pair_a, 3] == rq[pair_b, 3]
prefix_depth = 1 + eq2.astype(int) + (eq2 & eq3).astype(int) + (eq2 & eq3 & eq4).astype(int)

# Random pairs give the depth-0 bin and the marginal-confound bins.
pair_rng = np.random.default_rng(0)
rp = pair_rng.integers(0, N_CATALOG, size=(400_000, 2))
rp = rp[rp[:, 0] != rp[:, 1]]
cos_rand = (E[rp[:, 0]] * E[rp[:, 1]]).sum(axis=1)
r_eq = rq[rp[:, 0]] == rq[rp[:, 1]]
baseline = float(cos_rand.mean())

rows = [{"pairs share": "nothing at level 1 (depth 0)",
         "mean cosine": round(float(cos_rand[~r_eq[:, 0]].mean()), 4),
         "n pairs": int((~r_eq[:, 0]).sum())}]
for d, label in [(1, "exact prefix c1 (depth 1)"), (2, "exact prefix c1,c2 (depth 2)"),
                 (3, "exact prefix c1..c3 (depth 3)"), (4, "full RQ code c1..c4 (depth 4)")]:
    m = prefix_depth == d
    rows.append({"pairs share": label,
                 "mean cosine": round(float(cos_in[m].mean()), 4),
                 "n pairs": int(m.sum())})
for l, label in [(1, "same c2 but DIFFERENT c1"), (2, "same c3 but DIFFERENT c1")]:
    m = r_eq[:, l] & ~r_eq[:, 0]
    rows.append({"pairs share": label,
                 "mean cosine": round(float(cos_rand[m].mean()), 4),
                 "n pairs": int(m.sum())})
print(f"\nMean behavioral cosine by EXACT shared prefix depth (baseline = {baseline:.4f}):")
print(pd.DataFrame(rows).to_string(index=False))

# Conditional geometry: within a level-1 cluster, does level-2 geometry grade?
sub = ~eq2                                                    # same c1, different c2
d2 = code_dist[1, rq[pair_a[sub], 1], rq[pair_b[sub], 1]]
qs = np.quantile(d2, [0.25, 0.5, 0.75])
cond_rows = [{"level-2 relation (within same c1)": "same c2 (depth >= 2)",
              "mean cosine": round(float(cos_in[eq2].mean()), 4), "n pairs": int(eq2.sum())}]
for lab, m in [("Q1 closest", d2 <= qs[0]), ("Q2", (d2 > qs[0]) & (d2 <= qs[1])),
               ("Q3", (d2 > qs[1]) & (d2 <= qs[2])), ("Q4 farthest", d2 > qs[2])]:
    cond_rows.append({"level-2 relation (within same c1)": lab,
                      "mean cosine": round(float(cos_in[sub][m].mean()), 4),
                      "n pairs": int(m.sum())})
print("\nWithin same level-1 cluster: level-2 codebook distance vs behavioral cosine:")
print(pd.DataFrame(cond_rows).to_string(index=False))

In [ ]:
# --- Prefix-gated hierarchical similarity as a reward --------------------------
def hier_similarity(weights):
    """S_hier to each user's target: levels contribute only while the prefix
    matches; the first divergent RQ level contributes its codebook-geometry
    similarity; deeper levels contribute 0. Collision suffix: exact match."""
    w = np.asarray(weights, dtype=np.float64)
    sims = np.zeros((N_USERS, N_CATALOG))
    for col in range(N_USERS):
        t = int(batch["target_item_ids"][col]) - 1            # catalog row of target
        active = np.ones(N_CATALOG)
        s = np.zeros(N_CATALOG)
        for level in range(SID_LEN):
            ca, ct = codes_cat[:, level], codes_cat[t, level]
            eq = (ca == ct).astype(np.float64)
            if level < N_RQ_LEVELS:
                grade = np.where(eq == 1.0, 1.0, 1.0 - code_dist[level, ca, ct])
            else:
                grade = eq
            s += w[level] * active * grade
            active = active * eq
        sims[col] = s / w.sum()
    return torch.from_numpy(sims).float()


hier_rewards = {
    "hierarchical · level-decay w": hier_similarity(LEVEL_WEIGHTS).clamp_min(1e-6),
    "hierarchical · equal w": hier_similarity(np.ones(SID_LEN)).clamp_min(1e-6),
}
rows = []
for name, r in hier_rewards.items():
    tgt = r[is_positive]
    pct = (r < tgt.unsqueeze(1)).float().mean(1) * 100
    contrast = tgt / r.median(dim=1).values
    rows.append({"variant": name,
        **{f"pct u{int(batch['user_ids'][i])}": round(float(p), 1) for i, p in enumerate(pct)},
        **{f"tgt/med u{int(batch['user_ids'][i])}": round(float(c), 2) for i, c in enumerate(contrast)},
        "logR std": round(float(torch.log(r).std()), 2)})
print("Hierarchical prefix-gated reward:")
print(pd.DataFrame(rows).to_string(index=False))

# Does gating help? Rank random pairs by similarity, compare with the flat
# (ungated) codebook variant from 5.3 on the same pairs.
def pairwise_sim(a, b, weights, gated):
    w = np.asarray(weights, dtype=np.float64)
    active = np.ones(len(a)); s = np.zeros(len(a))
    for level in range(SID_LEN):
        ca, cb = codes_cat[a, level], codes_cat[b, level]
        eq = (ca == cb).astype(np.float64)
        grade = (np.where(eq == 1.0, 1.0, 1.0 - code_dist[level, ca, cb])
                 if level < N_RQ_LEVELS else eq)
        s += w[level] * (active if gated else 1.0) * grade
        if gated:
            active = active * eq
    return s / w.sum()

from scipy.stats import spearmanr
s_gated = pairwise_sim(rp[:, 0], rp[:, 1], LEVEL_WEIGHTS, gated=True)
s_flat = pairwise_sim(rp[:, 0], rp[:, 1], LEVEL_WEIGHTS, gated=False)
print("\nspearman with behavioral cosine over random pairs:")
print(f"  prefix-gated:  {spearmanr(s_gated[:100_000], cos_rand[:100_000]).statistic:.4f}")
print(f"  flat codebook: {spearmanr(s_flat[:100_000], cos_rand[:100_000]).statistic:.4f}")
top_g = np.argsort(-s_gated)[:4000]
top_f = np.argsort(-s_flat)[:4000]
print(f"  mean cosine of top-1% pairs: gated {cos_rand[top_g].mean():.4f} "
      f"vs flat {cos_rand[top_f].mean():.4f}")

In [ ]:
# --- Plot: reward = hierarchical prefix-gated similarity, full catalog ---------
hier_variant_names = list(hier_rewards)
fig, axes = plt.subplots(
    len(hier_variant_names), N_USERS,
    figsize=(3.4 * N_USERS, 2.4 * len(hier_variant_names)),
    sharex="row", sharey="row",
)
for row, name in enumerate(hier_variant_names):
    r_all = hier_rewards[name].numpy()
    bins = np.linspace(0, max(r_all.max(), 1.0) * 1.001, 60)
    for col in range(N_USERS):
        ax = axes[row, col]
        r = r_all[col]
        ax.hist(r, bins=bins, color=C_VERIFIED, edgecolor=SURFACE,
                linewidth=0.3, zorder=3)
        ax.set_yscale("log")
        tgt = float(r[is_positive[col].numpy()][0])
        pct = float((r < tgt).mean() * 100)
        ax.axvline(tgt, color=C_UNVERIFIED, linewidth=1.6, zorder=4)
        ax.text(0.98, 0.92, f"target: {tgt:.3g}\npct {pct:.1f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=8, color=C_UNVERIFIED)
        ax.grid(axis="x", visible=False)
        if row == 0:
            rt = batch["target_ratings"][col].item()
            ver = "verified" if bool(batch["target_verified_purchase"][col]) else "unverified"
            ax.set_title(f"user {int(batch['user_ids'][col])} · target {rt:.0f}★ {ver}",
                         fontsize=9, color=INK_2)
        if col == 0:
            ax.set_ylabel(name.replace(" · ", "\n"), fontsize=9, color=INK)

fig.suptitle(
    "Reward = hierarchical prefix-gated similarity to the positive target "
    "(levels count only while the prefix matches; orange line = target)",
    fontweight="bold", x=0.01, ha="left",
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

### Reading the hierarchical analysis — the levels are interdependent, and it matters

- **The hierarchy is real.** Behavioral coherence climbs monotonically with
  exact prefix depth: **0.054 (depth 0) → 0.131 (c1) → 0.178 (c1,c2) → 0.204
  (c1..c3) → 0.305 (full RQ code)** — depth-4 pairs (the pre-collision groups,
  132 pairs) are the most behaviorally coherent bin in the entire analysis.
  Deeper levels *do* carry signal; 5.3 just measured them out of context.
- **The marginal confound is confirmed.** Pairs sharing a level-2 token
  *without* sharing level 1 sit exactly at the random baseline (0.0548, and
  0.0509 for level 3) — averaging those pairs into the per-level bins is what
  made levels 2–4 look empty in the marginal check.
- **Even conditionally, deep geometry does not grade.** Within a level-1
  cluster, an exact level-2 match jumps to 0.183, but the codebook distance
  between *different* level-2 codes is non-monotone (0.135 / 0.126 / 0.124 /
  0.140) — beyond level 1, information lives in exact matches, not in the
  residual codebook geometry. So the geometry grading in $S_{\text{hier}}$
  earns its keep at the level-1 divergence (where 5.3 proved it orders items)
  and is mostly neutral deeper down.
- **Prefix gating improves the reward's ordering.** Over random pairs,
  spearman with behavioral cosine rises from 0.108 (flat codebook Gower, 5.3)
  to **0.136** with the gate, and the top-1% most-similar pairs under the gated
  score are more coherent (0.118 vs 0.107).
- **Gating inverts the weight argument.** In the *flat* formulation, decaying
  weights were needed because deep levels were marginal noise. Under the
  prefix gate, a deep contribution only exists when the whole prefix already
  matches — and those pairs are the most informative of all — so **equal
  weights now give the sharper reward**: target/median contrast 6.3–9.8
  (vs 2.4–3.8 with decay) at the same `log R` std ≈ 0.22. The gate itself
  encodes the hierarchy; the weights no longer need to.
- The reward remains target-conditioned (percentile 100 by construction), with
  the usual consequence: it says "close to what the user bought", and its
  natural use is as a graded modulator or in a product with a collaborative
  signal. Among all token-only similarities in this notebook, **hierarchical ·
  equal w is the best-founded**: densest signal-bearing shoulder, strongest
  pair-ranking agreement with behavior, and every modeling choice in it is
  backed by a check above.

#### 5.5.1 Why doesn't level-2 geometry grade? Codebook vs true residuals

The finding above — within a level-1 cluster, an exact level-2 match jumps but
the *distance* between different level-2 codes is non-monotone — has two
competing explanations:

- **(a) The level-2 codebook is badly organized.** The level-2 codebook is
  shared across all level-1 clusters, so codes that are far apart in latent
  space might still quantize behaviorally similar residuals — the geometry of
  the *codes* would then misrepresent the geometry of the *residuals*.
- **(b) The residual scale itself carries no behavioral signal.** Once the
  level-1 cluster is fixed, the remaining variation (what levels 2+ encode)
  is reconstruction detail that user behavior cannot see — then *no* codebook
  organization could grade it.

They are distinguishable: run the **RQ-VAE encoder** on the content embeddings
to get each item's latent $z$ and its **true, unquantized residual**
$r_1 = z - \mathbf{e}_1(c_1)$. If explanation (a) holds, the *true* residual
distance $\lVert r_{1,a} - r_{1,b} \rVert$ should grade behavioral similarity
even though the quantized one doesn't. If (b) holds, both are equally flat.
The cell validates the encoder pipeline by re-encoding the catalog and
checking the codes against the stored semantic IDs, then compares three decile
curves against the SASRec oracle: level-1 codebook distance (the graded
reference), quantized level-2 distance, and true residual distance.
(Reuses `Z` from 5.4 and the pair sets from 5.5.)

In [ ]:
# --- Quantized level-2 distance vs the TRUE residual geometry ------------------
from flowrec.item_representation.semantic_id.rqvae import RQVAE, RQVAEConfig

rq_payload = torch.load(RQVAE_DIR / "model.pt", map_location="cpu", weights_only=False)
rqvae = RQVAE(RQVAEConfig(**rq_payload["rqvae_config"]))
rqvae.load_state_dict(rq_payload["model_state_dict"])
rqvae.eval()
with torch.no_grad():
    z_lat = rqvae.encoder(Z)                                   # [N_CATALOG, latent_dim]
    codes_pred = rqvae.quantizer.encode(z_lat).numpy()
match = (codes_pred == rq).all(axis=1).mean()
print(f"encoder sanity: re-encoded codes match stored semantic IDs for {match:.1%} of items")
assert match > 0.999, "encoder pipeline does not reproduce the stored semantic IDs"

e1 = codebooks[0][torch.tensor(rq[:, 0])]                      # level-1 centroid per item
r1 = (z_lat - e1).numpy()                                      # true residual after level 1
r1_scale = float(np.linalg.norm(r1, axis=1).mean())
z_scale = float(z_lat.norm(dim=1).mean())
print(f"scale: mean ||z|| = {z_scale:.3f}, mean ||r1|| = {r1_scale:.3f} "
      f"(residual = {r1_scale / z_scale:.0%} of the latent)")

d_code2 = code_dist[1, rq[pair_a, 1], rq[pair_b, 1]]           # quantized level-2 distance
d_res = np.linalg.norm(r1[pair_a] - r1[pair_b], axis=1)        # true residual distance
d_code1_rand = code_dist[0, rq[rp[:, 0], 0], rq[rp[:, 1], 0]]  # level-1 distance, random pairs

from scipy.stats import spearmanr
print("\nspearman with behavioral cosine (within same level-1 cluster):")
print(f"  quantized level-2 codebook distance:  {spearmanr(d_code2, cos_in).statistic:+.4f}")
print(f"  TRUE residual distance ||r1_a-r1_b||: {spearmanr(d_res, cos_in).statistic:+.4f}")
print(f"  (reference: level-1 codebook distance on random pairs "
      f"{spearmanr(d_code1_rand, cos_rand).statistic:+.4f})")
print(f"  corr(quantized level-2 dist, true residual dist): "
      f"{spearmanr(d_code2[~eq2], d_res[~eq2]).statistic:+.4f}")


def decile_curve(metric, cos_vals):
    """Mean behavioral cosine (± SEM) in deciles of `metric`."""
    edges = np.quantile(metric, np.linspace(0, 1, 11))
    xs, ys, es = [], [], []
    for k in range(10):
        m = (metric >= edges[k]) & ((metric <= edges[k + 1]) if k == 9 else (metric < edges[k + 1]))
        xs.append(float(metric[m].mean()))
        ys.append(float(cos_vals[m].mean()))
        es.append(float(cos_vals[m].std() / np.sqrt(max(int(m.sum()), 1))))
    return np.array(xs), np.array(ys), np.array(es)


curve_l1 = decile_curve(d_code1_rand, cos_rand)
curve_c2 = decile_curve(d_code2[~eq2], cos_in[~eq2])           # different-c2 pairs only
curve_res = decile_curve(d_res, cos_in)                        # all same-c1 pairs
same_c2_point = (float(cos_in[eq2].mean()), float(cos_in[eq2].std() / np.sqrt(eq2.sum())))
within_mean = float(cos_in.mean())
print(f"\nsame-c2 jump: {same_c2_point[0]:.4f} vs within-cluster mean {within_mean:.4f}")
print("level-1 decile curve (random pairs):", np.round(curve_l1[1], 4))
print("quantized level-2 curve (same c1):  ", np.round(curve_c2[1], 4))
print("true residual curve (same c1):      ", np.round(curve_res[1], 4))

In [ ]:
# --- Plot: does geometry grade similarity? three decile curves -----------------
fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.0))
panels = [
    ("Level 1 · all pairs\ncodebook distance", curve_l1, None),
    ("Level 2 · within same c₁\nquantized codebook distance", curve_c2, same_c2_point),
    ("Level 2 · within same c₁\nTRUE residual distance ‖Δr₁‖", curve_res, None),
]
for i, (ax, (title, (xs, ys, es), extra)) in enumerate(zip(axes, panels)):
    ax.axhline(baseline, color=MUTED, linewidth=1, linestyle="--", zorder=2)
    if i > 0:
        ax.axhline(within_mean, color=MUTED, linewidth=1, linestyle=":", zorder=2)
        ax.set_ylim(0.04, 0.20)
    ax.errorbar(xs, ys, yerr=es, color=C_VERIFIED, marker="o", markersize=5,
                linewidth=2, capsize=2, zorder=4)
    if extra is not None:
        ax.errorbar([0], [extra[0]], yerr=[extra[1]], color=C_UNVERIFIED, marker="D",
                    markersize=7, capsize=2, zorder=5)
        ax.annotate("same c₂", (0, extra[0]), textcoords="offset points",
                    xytext=(8, 4), fontsize=9, color=C_UNVERIFIED)
    ax.set_title(title, fontsize=10, color=INK_2)
    ax.set_xlabel("distance (decile mean)")
axes[0].annotate("random-pair baseline", (curve_l1[0][-1], baseline),
                 textcoords="offset points", xytext=(-2, 5), ha="right",
                 fontsize=8.5, color=MUTED)
axes[1].annotate("within-cluster mean", (curve_c2[0][-1], within_mean),
                 textcoords="offset points", xytext=(-2, 5), ha="right",
                 fontsize=8.5, color=MUTED)
axes[0].set_ylabel("mean behavioral cosine (SASRec)")

fig.suptitle(
    "Does geometry grade similarity? Level-1 codebook distance does; below it, neither "
    "the quantized nor the TRUE residual distance carries a monotone signal",
    fontweight="bold", x=0.01, ha="left",
)
fig.tight_layout(rect=(0, 0, 1, 0.90))
plt.show()

#### Reading the residual-geometry analysis — explanation (b) wins

- **The pipeline is exact:** re-encoding the content embeddings reproduces the
  stored semantic IDs for 100% of items, so the residuals analyzed here are
  precisely what the level-2 codebook quantizes.
- **The codebook is blameless.** The quantized level-2 distance tracks the true
  residual distance faithfully (spearman +0.76) — and *both* are equally
  uninformative about behavior: spearman with behavioral cosine is −0.005
  (quantized) and −0.012 (true residual), against **−0.133** for level-1
  distance. Even a perfect, un-quantized view of the residual geometry would
  not grade similarity: **the residual scale itself (mean ‖r₁‖ ≈ 18% of the
  latent norm) carries no monotone behavioral signal.** That is explanation
  (b): levels 2+ encode reconstruction detail that user behavior cannot see.
- **The U-shape is real, and both curves share it** — flat middle (~0.12) with
  mild elevation at *both* extremes. The close end (closest decile 0.156,
  same-c₂ jump 0.183) is near-duplicate identity: essentially-the-same items.
  The far end (~0.15) are pairs of large-residual items — items the level-1
  centroid fits worst, i.e. cluster-boundary items, which plausibly share
  "atypical for this cluster" character. Neither extreme is a gradient a
  Gower-style `1 − d/range` term can exploit.
- **Consequence for the reward:** grading by geometry is justified exactly
  once — at the **level-1 divergence**. Below level 1, similarity evidence is
  *discrete* (exact prefix matches, and the deeper the rarer and stronger:
  0.131 → 0.178 → 0.204 → 0.305), so the hierarchical reward's graded term at
  deep divergences is neutral at best and could be replaced by exact-match
  gating alone without loss. This also sharpens the interpretation of the
  semantic ID itself: level 1 is a *behaviorally meaningful* coarse geometry;
  levels 2–4 are an *identity code* within the cluster.

#### 5.5.2 The same checks under the content oracle

All informativeness checks so far used the **behavioral** oracle (tuned-SASRec
item embeddings). But that conflates two different questions: *does a level's
distance encode content structure?* and *does that structure matter for what
users buy next?* Here we repeat the per-level analysis with the **semantic
(content) oracle** — the cosine between the items' 384-d MiniLM content
embeddings (`Zn` from 5.4, the very space the RQ-VAE compresses):

- the two per-level tables from 5.3 (mean pair cosine by token **index**
  distance and by **codebook-vector** distance), oracle swapped;
- the prefix-depth ladder from 5.5 under the content oracle;
- the conditional level-2 decile curves from 5.5.1 (quantized codebook
  distance and TRUE residual distance within a level-1 cluster).

Comparing the two oracles separates the semantic ID's *representation quality*
(does it faithfully encode content geometry?) from its *behavioral relevance*
(does that geometry predict co-consumption?). All pair sets, seeds, and
binning recipes are identical to the earlier sections, so every number is
directly comparable.

In [ ]:
# --- Per-level informativeness, oracle = content-embedding cosine --------------
Znp = Zn.numpy()
sem_in = (Znp[pair_a] * Znp[pair_b]).sum(axis=1)               # within-c1 pairs
sem_rand = (Znp[rp[:, 0]] * Znp[rp[:, 1]]).sum(axis=1)         # random pairs
sem_baseline = float(sem_rand.mean())

IDX_BINS = [("|Δ| = 0", 0, 0), ("|Δ| = 1", 1, 1), ("|Δ| 2-5", 2, 5),
            ("|Δ| 6-20", 6, 20), ("|Δ| > 20", 21, 10**9)]
idx_rows, cb_rows = [], []
for level in range(N_RQ_LEVELS):
    a, b = rq[rp[:, 0], level], rq[rp[:, 1], level]
    d_idx = np.abs(a - b)
    idx_rows.append({"level": level + 1,
        **{lab: round(float(sem_rand[(d_idx >= lo) & (d_idx <= hi)].mean()), 4)
           for lab, lo, hi in IDX_BINS}})
    d_cb = code_dist[level, a, b]
    nz = d_cb[d_cb > 0]
    qs = np.quantile(nz, [0.25, 0.5, 0.75])
    cb_bins = [("dist = 0", d_cb == 0), ("Q1 (closest)", (d_cb > 0) & (d_cb <= qs[0])),
               ("Q2", (d_cb > qs[0]) & (d_cb <= qs[1])), ("Q3", (d_cb > qs[1]) & (d_cb <= qs[2])),
               ("Q4 (farthest)", d_cb > qs[2])]
    cb_rows.append({"level": level + 1,
        **{lab: round(float(sem_rand[m].mean()), 4) for lab, m in cb_bins}})
print(f"Mean CONTENT-embedding cosine of item pairs (baseline = {sem_baseline:.4f})")
print("\nby token INDEX distance:")
print(pd.DataFrame(idx_rows).to_string(index=False))
print("\nby CODEBOOK-vector distance (quartiles of nonzero distances):")
print(pd.DataFrame(cb_rows).to_string(index=False))

# Prefix-depth ladder under the content oracle (same pair sets as 5.5).
ladder = [{"pairs share": "nothing at level 1 (depth 0)",
           "mean content cos": round(float(sem_rand[~r_eq[:, 0]].mean()), 4),
           "n pairs": int((~r_eq[:, 0]).sum())}]
for d, label in [(1, "exact prefix c1 (depth 1)"), (2, "exact prefix c1,c2 (depth 2)"),
                 (3, "exact prefix c1..c3 (depth 3)"), (4, "full RQ code c1..c4 (depth 4)")]:
    m = prefix_depth == d
    ladder.append({"pairs share": label,
                   "mean content cos": round(float(sem_in[m].mean()), 4),
                   "n pairs": int(m.sum())})
m = r_eq[:, 1] & ~r_eq[:, 0]
ladder.append({"pairs share": "same c2 but DIFFERENT c1",
               "mean content cos": round(float(sem_rand[m].mean()), 4),
               "n pairs": int(m.sum())})
print("\nPrefix-depth ladder under the content oracle:")
print(pd.DataFrame(ladder).to_string(index=False))

# Conditional level-2 geometry under the content oracle (distances from 5.5.1).
from scipy.stats import spearmanr
print("\nspearman with CONTENT cosine (vs the behavioral values in parentheses):")
print(f"  level-1 codebook distance (random pairs):        "
      f"{spearmanr(d_code1_rand, sem_rand).statistic:+.4f}  (behavioral -0.133)")
print(f"  level-2 quantized distance (within same c1):     "
      f"{spearmanr(d_code2, sem_in).statistic:+.4f}  (behavioral -0.005)")
print(f"  level-2 TRUE residual distance (within same c1): "
      f"{spearmanr(d_res, sem_in).statistic:+.4f}  (behavioral -0.012)")

sem_curve_l1 = decile_curve(d_code1_rand, sem_rand)
sem_curve_c2 = decile_curve(d_code2[~eq2], sem_in[~eq2])
sem_curve_res = decile_curve(d_res, sem_in)
sem_same_c2 = (float(sem_in[eq2].mean()), float(sem_in[eq2].std() / np.sqrt(eq2.sum())))
sem_within = float(sem_in.mean())
print(f"\nsame-c2 jump: {sem_same_c2[0]:.4f} vs within-cluster mean {sem_within:.4f}")

In [ ]:
# --- Plot: the same three decile curves, content oracle ------------------------
fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.0))
panels = [
    ("Level 1 · all pairs\ncodebook distance", sem_curve_l1, None),
    ("Level 2 · within same c₁\nquantized codebook distance", sem_curve_c2, sem_same_c2),
    ("Level 2 · within same c₁\nTRUE residual distance ‖Δr₁‖", sem_curve_res, None),
]
for i, (ax, (title, (xs, ys, es), extra)) in enumerate(zip(axes, panels)):
    ax.axhline(sem_baseline, color=MUTED, linewidth=1, linestyle="--", zorder=2)
    if i > 0:
        ax.axhline(sem_within, color=MUTED, linewidth=1, linestyle=":", zorder=2)
        ax.set_ylim(0.22, 0.82)
    ax.errorbar(xs, ys, yerr=es, color=C_VERIFIED, marker="o", markersize=5,
                linewidth=2, capsize=2, zorder=4)
    if extra is not None:
        ax.errorbar([0], [extra[0]], yerr=[extra[1]], color=C_UNVERIFIED, marker="D",
                    markersize=7, capsize=2, zorder=5)
        ax.annotate("same c₂", (0, extra[0]), textcoords="offset points",
                    xytext=(8, 4), fontsize=9, color=C_UNVERIFIED)
    ax.set_title(title, fontsize=10, color=INK_2)
    ax.set_xlabel("distance (decile mean)")
axes[0].annotate("random-pair baseline", (sem_curve_l1[0][-1], sem_baseline),
                 textcoords="offset points", xytext=(-2, 5), ha="right",
                 fontsize=8.5, color=MUTED)
axes[1].annotate("within-cluster mean", (sem_curve_c2[0][-1], sem_within),
                 textcoords="offset points", xytext=(-2, 5), ha="right",
                 fontsize=8.5, color=MUTED)
axes[0].set_ylabel("mean content cosine (MiniLM)")

fig.suptitle(
    "The content oracle sees what the behavioral one cannot: every level's geometry "
    "grades content similarity — compare with the flat behavioral curves in 5.5.1",
    fontweight="bold", x=0.01, ha="left",
)
fig.tight_layout(rect=(0, 0, 1, 0.90))
plt.show()

#### Reading the content-oracle comparison — representation vs relevance

- **Under the content oracle, everything grades.** Codebook distance is
  monotone at *every* level (level 1: 0.554 → 0.357 → 0.294 → 0.270 → 0.249;
  levels 2–4 weakly but consistently), and within a level-1 cluster the
  conditional level-2 curves are strongly graded: spearman **−0.41**
  (quantized) and **−0.52** (true residual) with content cosine — the same
  distances that scored −0.005 / −0.012 against behavior in 5.5.1.
- **The prefix ladder is dramatic:** 0.29 (depth 0) → 0.545 → 0.755 → 0.921 →
  **0.985** for full-RQ-code pairs — items sharing the whole code are
  near-identical in content. Together with 5.5, this pins the interpretation:
  **deeper levels encode genuine, hierarchically refined content structure —
  it is simply invisible to behavior.** Content detail below the level-1
  granularity does not change what users buy next.
- **Index distance is nominal even for content.** |Δ| = 1 pairs sit at the
  content baseline at every level (level 1: 0.281 vs baseline 0.293) — the
  definitive refutation of the "nearby indices mean something" assumption:
  it fails against *both* oracles, marginal and conditional alike.
- **The marginal confound replicates under content too:** same c₂ with
  different c₁ is ≈ baseline (0.306), confirming that level-2 codes only mean
  something inside their level-1 context — for content just as for behavior.
- **Design consequence.** For a *content*-similarity reward, the semantic ID
  supports full graded use: prefix gating + geometry grading at every level
  (the 5.5 hierarchical reward would be well-founded end-to-end against this
  oracle). For a *behavioral* reward — which is what TB training on next-item
  prediction ultimately optimizes — the 5.5.1 conclusion stands: geometry at
  the level-1 divergence, discrete prefix evidence below. The gap between the
  two oracles is itself the finding: the RQ-VAE compresses content faithfully,
  and content stops predicting behavior below the coarse-cluster scale.

### 5.6 Synthesis — independent vs hierarchical token analysis, both oracles

One figure condensing the whole investigation. Two ways of asking "does sharing
semantic-ID structure at level $\ell$ mean the items are similar?":

- **independent** (sections 5.3 / 5.5.2): pairs sharing the *token at level
  $\ell$*, regardless of the other levels — the marginal, per-level view;
- **hierarchical** (section 5.5): pairs sharing the *exact prefix
  $c_1..c_\ell$* — the view that respects the RQ residual structure.

Each is evaluated against both similarity oracles: **behavioral** (tuned-SASRec
item embeddings, trained on co-consumption, blind to semantic IDs) and
**content** (the MiniLM embeddings the RQ-VAE compresses). The ✕ marks the
marginal confound (same $c_2$, different $c_1$), and the third panel divides
each curve by its oracle's random-pair baseline to put both oracles on one
scale. Reuses the pair sets of 5.5 / 5.5.2 (same seeds, directly comparable).

In [ ]:
# --- Independent vs hierarchical, behavioral vs content: the synthesis figure --
levels = np.arange(1, N_RQ_LEVELS + 1)


def method_series(oracle_rand, oracle_in):
    """(independent per-level, ns, hierarchical per-depth, ns, confound)."""
    indep = np.array([float(oracle_rand[r_eq[:, l]].mean()) for l in range(N_RQ_LEVELS)])
    indep_n = np.array([int(r_eq[:, l].sum()) for l in range(N_RQ_LEVELS)])
    hier = np.array([float(oracle_in[prefix_depth == d].mean()) for d in levels])
    hier_n = np.array([int((prefix_depth == d).sum()) for d in levels])
    confound = float(oracle_rand[r_eq[:, 1] & ~r_eq[:, 0]].mean())
    return indep, indep_n, hier, hier_n, confound


beh = method_series(cos_rand, cos_in)      # behavioral oracle (5.5 pair sets)
sem = method_series(sem_rand, sem_in)      # content oracle (5.5.2)
comparison = pd.DataFrame({
    "level": levels,
    "indep · behavioral": beh[0].round(4), "hier · behavioral": beh[2].round(4),
    "indep · content": sem[0].round(4), "hier · content": sem[2].round(4),
    "n hier pairs": beh[3],
})
print(f"baselines: behavioral {baseline:.4f}, content {sem_baseline:.4f}")
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.3))
C_HIER, C_INDEP = C_VERIFIED, C_UNVERIFIED

for ax, (indep, indep_n, hier, hier_n, confound), base, title in [
    (axes[0], beh, baseline, "Behavioral oracle\n(SASRec item embeddings)"),
    (axes[1], sem, sem_baseline, "Content oracle\n(MiniLM content embeddings)"),
]:
    ax.axhline(base, color=MUTED, linewidth=1, linestyle="--", zorder=2)
    ax.plot(levels, indep, marker="o", markersize=6, linewidth=2, color=C_INDEP,
            label="independent: same token at level ℓ", zorder=4)
    ax.plot(levels, hier, marker="o", markersize=6, linewidth=2, color=C_HIER,
            label="hierarchical: same prefix c₁..cₗ", zorder=4)
    ax.scatter([2], [confound], marker="x", s=70, color=MUTED, zorder=5)
    ax.annotate("same c₂, different c₁", (2, confound), textcoords="offset points",
                xytext=(12, -22), fontsize=8, color=MUTED)
    for x, y, n in zip(levels, hier, hier_n):
        ax.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=8, color=C_HIER)
        if n < 10_000:
            ax.annotate(f"n={n}", (x, y), textcoords="offset points", xytext=(0, -14),
                        ha="center", fontsize=7, color=MUTED)
    for x, y in zip(levels, indep):
        ax.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, -13),
                    ha="center", fontsize=8, color=C_INDEP)
    ax.set_xticks(levels)
    ax.set_xlabel("semantic-ID level ℓ")
    ax.set_title(title, fontsize=10, color=INK_2)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.18)
axes[0].set_ylabel("mean pair cosine")
axes[0].legend(frameon=False, fontsize=8.5, loc="upper left")
axes[0].annotate("random-pair baseline", (4, baseline), textcoords="offset points",
                 xytext=(-2, 5), ha="right", fontsize=8, color=MUTED)

ax = axes[2]
ax.axhline(1.0, color=MUTED, linewidth=1, linestyle="--", zorder=2)
for vals, base, color, style, label in [
    (beh[2], baseline, C_HIER, "-", "hier · behavioral"),
    (sem[2], sem_baseline, C_HIER, "--", "hier · content"),
    (beh[0], baseline, C_INDEP, "-", "indep · behavioral"),
    (sem[0], sem_baseline, C_INDEP, "--", "indep · content"),
]:
    ax.plot(levels, vals / base, marker="o", markersize=5, linewidth=2,
            color=color, linestyle=style, label=label, zorder=4)
ax.set_xticks(levels)
ax.set_xlabel("semantic-ID level ℓ")
ax.set_ylabel("lift over the oracle's baseline (×)")
ax.set_title("Both oracles on one scale\n(cosine ÷ baseline)", fontsize=10, color=INK_2)
ax.grid(axis="x", visible=False)
ax.legend(frameon=False, fontsize=8.5, loc="upper left")
ax.margins(y=0.15)

fig.suptitle(
    "Independent (per-level) vs hierarchical (prefix-conditional) token analysis",
    fontweight="bold", x=0.01, ha="left",
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

### Reading the synthesis

- **The two methods agree at level 1 and diverge everywhere below it.** At
  $\ell = 1$ they measure nearly the same thing (0.134 vs 0.131 behavioral,
  0.552 vs 0.545 content). From level 2 on, the independent view collapses to
  the baseline under both oracles (behavioral 0.052–0.059 ≈ 0.055; content
  0.30–0.32 ≈ 0.29), while the hierarchical view climbs monotonically —
  behavioral 0.131 → 0.178 → 0.204 → 0.305, content 0.545 → 0.755 → 0.921 →
  0.985. The ✕ (same $c_2$, different $c_1$) sits exactly on the baseline in
  both panels: a deep token match without its prefix means *nothing*, which is
  precisely what the marginal analysis was averaging over.
- **The lift panel puts both oracles on one scale** and adds a twist: relative
  to its own baseline, the prefix ladder is *steeper behaviorally* (5.6× at
  full depth) than in content space (3.4×). Content similarity saturates —
  full-code pairs are near-identical text (0.985 cosine, ceiling ≈ 1) — while
  behavioral similarity keeps rising with every matched level. Deep prefix
  matches are rare (n = 6,259 / 589 / 132), but each additional matched level
  roughly doubles-to-triples the behavioral lift.
- **Takeaway for reward design, in one sentence:** semantic-ID levels are
  worthless independently but powerful cumulatively — a similarity reward must
  read the ID as a *prefix tree* (as $S_{\text{hier}}$ does), not as a bag of
  tokens (as the 5.2 positionwise similarity and every flat Gower variant do).

## 6. Conclusion — final reward portfolio

### Individual rewards

| reward | role | essence |
|---|---|---|
| `sasrec` · softplus | behavioral prior | *"plausible next item?"* — dense, smooth, no cliffs |
| `amazon_log` · item_centered | evidence modulator. Multiplier `clip(exp(β·w·z))` around 1.0, where `z` is the item-mean-centered rating residual and `w` down-weights unverified evidence | *"was the interaction good?"* — ≈1.0 neutral, trust-weighted, never standalone |
| `similarity` | target shaping | *"close to what actually happened?"* — graded partial credit, γ sharpens |

### Similarity modes

| mode | idea | verdict |
|---|---|---|
| `exact` | matched tokens / L | near-binary — baseline only |
| `hierarchical` | prefix-gated + level-1 geometry | ✅ **best-founded** (ladder 0.13 → 0.31) |
| `embedding_cosine` | cosine of content embeddings | smooth, global, weak |

*Rejected:* index distance · attribute Gower · embedding L2 · deep-level
geometry · `logged` / `binary` · `identity` / `minmax_batch` / `rank_percentile`

### Composition

| operator | formula | TB-safe | verdict |
|---|---|---|---|
| product · `"multiply"` | $R = \prod_i R_i^{w_i}$ | ✅ | **recommended** |
| sum · `"mix"` | $R = \sum_i w_i\,\text{norm}(R_i)$ | only `norm="none"` | ablation arm |

$$
R \;=\; \underbrace{R_{\text{sasrec}}}_{\text{plausible?}} \times
\underbrace{R_{\text{ic}}}_{\text{good?}} \times
\underbrace{S_{\text{hier}}^{\,\gamma}}_{\text{close?}}
$$

Verdict pending: the 3-seed × 11-variant ablation (`scripts/rewards_ablation.py`).

## 7. TIGER fine-tuning results — comparing the rewards

The verdict on section 6: test metrics of GFlowNet (TB) fine-tuning of the SFT
TIGER checkpoint under each reward, from the 3-seed cluster ablation
(`scripts/rewards_ablation.py`, results synced to
`experiments/rewards_ablation_cluster/`). The cell below aggregates every
per-run `summary.csv` (latest run wins per dataset/reward/seed), prints the
mean ± std table, and draws one figure per dataset: small dots = individual
seeds, large marker = mean, dashed line = the SFT baseline before any
fine-tuning, color = reward family (individual / product / sum). Rows are
sorted by mean NDCG@10. Re-run the cell after syncing new results — it always
reflects whatever runs have completed.

In [ ]:
# --- Cluster ablation results: aggregate + compare rewards ---------------------
import json
import sys

if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import matplotlib
_notebook_backend = matplotlib.get_backend()
import rewards_ablation as ra          # sets Agg on import - restore inline below
plt.switch_backend(_notebook_backend)

RESULTS_DIR = PROJECT_ROOT / "experiments" / "rewards_ablation_cluster"
run_frames = [
    frame
    for p in sorted(RESULTS_DIR.glob("*_rewards_ablation/summary.csv"))
    if not (frame := pd.read_csv(p)).empty
]
if not run_frames:
    raise FileNotFoundError(f"No synced run summaries under {RESULTS_DIR}")
results = (
    pd.concat(run_frames, ignore_index=True)
    .query("status == 'completed'")
    .drop_duplicates(subset=["dataset", "reward", "seed"], keep="last")
)
# Same-second run dirs could overwrite each other's summary.csv; recover any
# missing (dataset, reward, seed) from the per-run test_metrics.json.
have = set(map(tuple, results[["dataset", "reward", "seed"]].itertuples(index=False)))
recovered = []
for tm_path in sorted(RESULTS_DIR.glob("*_rewards_ablation/*/*/seed_*/test_metrics.json")):
    payload = json.loads(tm_path.read_text())
    seed = int(tm_path.parent.name.removeprefix("seed_"))
    key = (payload.get("dataset"), payload.get("reward"), seed)
    if key in have:
        continue
    recovered.append({"dataset": key[0], "reward": key[1], "seed": seed,
                      "status": "completed",
                      **{k: float(v) for k, v in payload.items()
                         if isinstance(v, (int, float))}})
    have.add(key)
if recovered:
    print(f"[results] recovered {len(recovered)} runs from test_metrics.json")
    results = pd.concat([results, pd.DataFrame(recovered)], ignore_index=True)

METRICS = ["HR@10", "NDCG@10", "Coverage@10"]
agg = (
    results[results["reward"] != "sft_baseline"]
    .groupby(["dataset", "reward"])[METRICS]
    .agg(["mean", "std", "count"])
    .round(4)
)
print("Completed runs per (dataset, reward):")
print(agg.to_string())


@torch.no_grad()
def sasrec_teacher_metrics(dataset, ks=(5, 10, 20), device="cpu"):
    """Test metrics of the tuned SASRec teacher itself (top-K by score, cached)."""
    cache_path = RESULTS_DIR / "sasrec_reference.json"
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    if dataset in cache:
        return cache[dataset]
    cfgs = ra.load_exp_configs(ra.path_from_repo("configs/exp1-standard params"))
    semantic_dir = ra.path_from_repo(cfgs["semantic"]["semantic_dir"])
    dm_t = ra.build_datamodule(
        dataset, cfgs["dataset"],
        semantic_id_path=semantic_dir / dataset / "semantic_ids.json",
        seed=int(cfgs["trainer"].get("seed", 43)), num_workers=0,
    )
    teacher = ra.load_sasrec_teacher(
        ra.find_sasrec_checkpoint(
            ra.path_from_repo("checkpoints/exp2-tuned-params/sasrec") / dataset), dm_t)
    teacher.to(torch.device(device)).eval()
    max_k = max(ks)
    hits = {k: 0.0 for k in ks}; ndcgs = {k: 0.0 for k in ks}
    seen = {k: set() for k in ks}; n = 0
    for tb in dm_t.test_dataloader():
        tb = ra.move_batch(tb, torch.device(device))
        scores = teacher.score_items(tb).float()
        scores[:, 0] = -float("inf")
        top = scores.topk(max_k, dim=1).indices.cpu()
        tgt = tb["target_item_ids"].long().cpu()
        match = top.eq(tgt.unsqueeze(1))
        n += int(tgt.numel())
        for k in ks:
            m = match[:, :k]
            hit = m.any(dim=1)
            rank = m.float().argmax(dim=1)
            disc = torch.zeros_like(hit, dtype=torch.float32)
            disc[hit] = 1.0 / torch.log2(rank[hit].float() + 2.0)
            hits[k] += float(hit.sum()); ndcgs[k] += float(disc.sum())
            seen[k].update(top[:, :k].reshape(-1).tolist())
    out = {"num_test_examples": float(n)}
    for k in ks:
        out[f"HR@{k}"] = hits[k] / max(n, 1)
        out[f"NDCG@{k}"] = ndcgs[k] / max(n, 1)
        out[f"Coverage@{k}"] = len(seen[k]) / max(int(dm_t.num_items), 1)
    cache[dataset] = out
    cache_path.write_text(json.dumps(cache, indent=2))
    return out


def reward_family(name):
    if "_x_" in name:
        return "product"
    if "_plus_" in name:
        return "sum"
    return "individual"


FAMILY_COLORS = {"individual": "#2a78d6", "product": "#eb6834", "sum": "#4a3aa7"}
C_TEACHER = "#008300"

for dataset, ds in results.groupby("dataset"):
    rewards_ds = ds[ds["reward"] != "sft_baseline"]
    if rewards_ds.empty:
        continue
    order = rewards_ds.groupby("reward")["NDCG@10"].mean().sort_values().index.tolist()
    baseline_row = ds[ds["reward"] == "sft_baseline"]
    teacher = sasrec_teacher_metrics(dataset)

    fig, axes = plt.subplots(
        1, len(METRICS),
        figsize=(4.6 * len(METRICS), 0.42 * len(order) + 1.8), sharey=True,
    )
    ypos = np.arange(len(order))
    for ax, metric in zip(axes, METRICS):
        if not baseline_row.empty:
            ax.axvline(float(baseline_row[metric].iloc[0]), color=MUTED,
                       linewidth=1.2, linestyle="--", zorder=2)
        teacher_val = float(teacher[metric])
        ax.axvline(teacher_val, color=C_TEACHER, linewidth=1.2, linestyle="-.", zorder=2)
        ax.annotate(f"{teacher_val:.3f}", xy=(teacher_val, 0.99), xycoords=("data", "axes fraction"),
                    ha="right", va="top", rotation=90, fontsize=7.5, color=C_TEACHER,
                    xytext=(-3, 0), textcoords="offset points")
        for y, reward in zip(ypos, order):
            rows = rewards_ds[rewards_ds["reward"] == reward]
            color = FAMILY_COLORS[reward_family(reward)]
            ax.scatter(rows[metric], [y] * len(rows), s=26, color=color,
                       alpha=0.45, zorder=3, linewidths=0)
            ax.scatter([float(rows[metric].mean())], [y], s=90, color=color,
                       edgecolor=SURFACE, linewidth=1.2, zorder=4)
        ax.set_title(metric, fontsize=10, color=INK_2)
        ax.grid(axis="y", visible=False)
        ax.tick_params(axis="x", labelsize=8)
    axes[0].set_yticks(ypos)
    axes[0].set_yticklabels(
        [f"{r}  (n={int((rewards_ds['reward'] == r).sum())})" for r in order],
        fontsize=9,
    )
    handles = [plt.Line2D([], [], marker="o", linestyle="", color=c, label=f)
               for f, c in FAMILY_COLORS.items()]
    handles += [plt.Line2D([], [], color=MUTED, linestyle="--", label="SFT baseline"),
                plt.Line2D([], [], color=C_TEACHER, linestyle="-.", label="SASRec teacher")]
    axes[0].legend(handles=handles, frameon=False, fontsize=8, loc="lower right")
    fig.suptitle(
        f"TIGER GFlowNet fine-tuning on {dataset} — test metrics by reward "
        "(dots = seeds, large marker = mean)",
        fontweight="bold", x=0.01, ha="left",
    )
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()